# GameTheory 15d - Valeur de Shapley de groupe : évaluer une équipe, pas une somme d'individus

**Navigation** : [<< 15-CooperativeGames (track principal)](GameTheory-15-CooperativeGames-Python.ipynb) | [Version Python](GameTheory-15d-Shapley-Groupes-Python.ipynb) | [Index](README.md) | [06b (Möbius, SocialChoice) <<](SocialChoice/ChoixSocial-06b-Mobius-Coalitions-Lean-Python.ipynb) | [Coalitions-01 (Pouvoir coalitionnel) <<](Coalitions/Coalitions-01-Pouvoir-Coalitionnel-SMT-Python.ipynb)

**Kernel** : .NET (C#), BCL uniquement

**Article distillé** : Flores R., Molina E., Tejada J. (2019). *Evaluating groups with the generalized Shapley value*. 4OR 17(2):141-172. doi:[10.1007/s10288-018-0380-8](https://doi.org/10.1007/s10288-018-0380-8)

***

## Introduction

Un décideur doit choisir **k agents** : former un binôme, recruter une équipe, cibler un groupe dans un réseau. Le réflexe est de calculer la valeur de Shapley de chacun et de garder les k meilleurs. Ce réflexe suppose que la valeur d'un groupe est la **somme** des valeurs de ses membres.

Ce notebook montre, calcul exact à l'appui, que cette hypothèse se trompe dès que les agents sont **redondants** (ils apportent la même chose) ou **complémentaires** (l'un rend l'autre utile). Il introduit le bon objet de mesure : la **valeur de Shapley de groupe** $\varphi^g(C)$, qui est la valeur de Shapley généralisée de Marichal, Kojadinovic et Fujimoto (2007), axiomatisée et appliquée à la sélection de groupes par Flores, Molina et Tejada.

| Section | Contenu |
|---|---|
| 1 | Le jeu fusionné de Lehrer, la définition de $\varphi^g$, l'axiome G-SPB face à la valeur additive |
| 2 | L'exemple 1 de l'article, reproduit en arithmétique exacte, et le théorème 2 |
| 3 | Profitabilité d'un groupe et complémentarité moyenne |
| 4 | Transposition au réseau des mariages florentins (Padgett) : calcul exact sur $2^{15}$ coalitions |
| 5 | Estimation Monte Carlo : calibration, puis club de karaté de Zachary ($2^{34}$ coalitions) |

**Prérequis** : GameTheory-15 (valeur de Shapley, axiomes). La décomposition de Möbius de SocialChoice 06b éclaire la section 3, sans être indispensable.

**Implémentation C#** : port BCL-only du carnet Python versionné, sans NuGet. `Rational` réduit chaque fraction par `BigInteger.GreatestCommonDivisor` ; les jeux sont représentés par des masques de bits. Les résultats déterministes sont assortis d'assertions exécutables. Les cellules Monte Carlo emploient `System.Random` : les derniers chiffres diffèrent donc du générateur NumPy, tandis que la convergence et les erreurs-types sont vérifiées.

In [1]:
// Noyau du port : BCL uniquement, arithmétique rationnelle exacte et graphes par masques.
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;
using System.Numerics;
using System.IO;
using System.Text;
using System.Diagnostics;

CultureInfo.DefaultThreadCurrentCulture = CultureInfo.InvariantCulture;
CultureInfo.CurrentCulture = CultureInfo.InvariantCulture;

readonly struct Rational : IComparable<Rational>, IEquatable<Rational>
{
    public static readonly Rational Zero = new Rational(BigInteger.Zero, BigInteger.One);
    public Rational(BigInteger n) : this(n, BigInteger.One) { }
    public BigInteger Num { get; }
    public BigInteger Den { get; }
    public Rational(BigInteger n, BigInteger d)
    {
        if (d.IsZero) throw new ArgumentException("Dénominateur nul.");
        if (d.Sign < 0) { n = -n; d = -d; }
        BigInteger g = BigInteger.GreatestCommonDivisor(BigInteger.Abs(n), d);
        if (!g.IsOne) { n /= g; d /= g; }
        Num = n; Den = d;
    }
    public static Rational operator +(Rational a, Rational b)
    {
        BigInteger g = BigInteger.GreatestCommonDivisor(a.Den, b.Den);
        BigInteger bd = b.Den / g, ad = a.Den / g;
        return new Rational(a.Num * bd + b.Num * ad, a.Den * bd);
    }
    public static Rational operator -(Rational a, Rational b) => a + new Rational(-b.Num, b.Den);
    public static Rational operator -(Rational a) => new Rational(-a.Num, a.Den);
    public static Rational operator *(Rational a, Rational b)
    {
        BigInteger g1 = BigInteger.GreatestCommonDivisor(BigInteger.Abs(a.Num), b.Den);
        BigInteger g2 = BigInteger.GreatestCommonDivisor(BigInteger.Abs(b.Num), a.Den);
        return new Rational((a.Num / g1) * (b.Num / g2), (a.Den / g2) * (b.Den / g1));
    }
    public static Rational operator /(Rational a, Rational b) => a * new Rational(b.Den, b.Num);
    public static Rational operator *(Rational a, int b) => a * new Rational(b, BigInteger.One);
    public static bool operator ==(Rational a, Rational b) => a.Num == b.Num && a.Den == b.Den;
    public static bool operator !=(Rational a, Rational b) => !(a == b);
    public static bool operator <(Rational a, Rational b) => a.Num * b.Den < b.Num * a.Den;
    public static bool operator >(Rational a, Rational b) => b < a;
    public double ToDouble() => (double)Num / (double)Den;
    public int CompareTo(Rational other) => (Num * other.Den).CompareTo(other.Num * Den);
    public override string ToString() => Den.IsOne ? Num.ToString() : $"{Num}/{Den}";
    public override bool Equals(object obj) => obj is Rational r && this == r;
    public bool Equals(Rational other) => this == other;
    public override int GetHashCode() => HashCode.Combine(Num, Den);
}
static Rational R(int n, int d = 1) => new Rational(n, d);
static void Require(bool condition, string message)
{
    if (!condition) throw new InvalidOperationException(message);
}
static void RequireExact(Rational actual, Rational expected, string label)
{
    if (actual != expected) throw new InvalidOperationException($"{label} : {actual}, attendu {expected}.");
}
static void RequireNear(double actual, double expected, double tolerance, string label)
{
    if (!double.IsFinite(actual) || Math.Abs(actual - expected) > tolerance)
        throw new InvalidOperationException($"{label} : {actual:G12}, attendu {expected} ± {tolerance}.");
}
static string Fmt(double x, int digits = 4) => x.ToString("F" + digits, CultureInfo.InvariantCulture);
static string Signed(double x) => x.ToString("+0.0000;-0.0000", CultureInfo.InvariantCulture);
static long Mask(params int[] players) => players.Aggregate(0L, (m, i) => m | (1L << i));
static int[] Indices(long mask, int n)
{
    var result = new List<int>();
    for (int i = 0; i < n; i++) if ((mask & (1L << i)) != 0) result.Add(i);
    return result.ToArray();
}
static int[] Idx(params int[] nodes) => nodes.Select(k => k - 1).ToArray();
static Rational[] ShapleyWeights(int n)
{
    var fact = new BigInteger[n + 1]; fact[0] = BigInteger.One;
    for (int i = 1; i <= n; i++) fact[i] = fact[i - 1] * i;
    return Enumerable.Range(0, n).Select(s => new Rational(fact[s] * fact[n - s - 1], fact[n])).ToArray();
}

class CoalitionGameCs
{
    public int N { get; }
    public string[] Names { get; }
    private readonly Func<long, Rational> characteristic;
    public CoalitionGameCs(int n, Func<long, Rational> value, string[] names = null)
    {
        N = n; Names = names ?? Enumerable.Range(0, n).Select(i => $"Player_{i}").ToArray();
        characteristic = value;
        Require(characteristic(0) == R(0), "La coalition vide doit valoir zéro.");
    }
    public Rational Value(long mask) => characteristic(mask);
}
static CoalitionGameCs ConnectivityGameCs(int n, (int A, int B)[] edges, string[] names = null)
{
    var adjacency = new long[n];
    foreach (var (a, b) in edges)
    {
        Require(a >= 0 && a < n && b >= 0 && b < n && a != b, "Arête invalide.");
        adjacency[a] |= 1L << b; adjacency[b] |= 1L << a;
    }
    Rational Value(long mask)
    {
        int size = BitOperations.PopCount((ulong)mask);
        if (size < 2) return R(0);
        int first = BitOperations.TrailingZeroCount((ulong)mask);
        long reached = 1L << first, frontier = reached;
        while (frontier != 0)
        {
            long next = 0, f = frontier;
            while (f != 0)
            {
                int node = BitOperations.TrailingZeroCount((ulong)f);
                next |= adjacency[node]; f &= f - 1;
            }
            frontier = next & mask & ~reached; reached |= frontier;
        }
        return R(reached == mask ? 1 : 0);
    }
    return new CoalitionGameCs(n, Value, names);
}
static Rational GroupValueCs(CoalitionGameCs game, long group)
{
    Require(group != 0 && (group >> game.N) == 0, "Groupe vide ou hors jeu.");
    Require(game.N <= 24, "Formule exacte limitée à 24 joueurs : employer MonteCarloCs.");
    int m = game.N - BitOperations.PopCount((ulong)group) + 1;
    var weights = ShapleyWeights(m); Rational total = R(0);
    for (int t = 0; t < (1 << game.N); t++)
        if ((t & group) == 0) total += weights[BitOperations.PopCount((uint)t)] * (game.Value((long)t | group) - game.Value(t));
    return total;
}
static (CoalitionGameCs Game, int[] Mapping) MergingGameCs(CoalitionGameCs game, long group)
{
    var rest = Enumerable.Range(0, game.N).Where(i => (group & (1L << i)) == 0).ToArray();
    int proxy = rest.Length;
    Rational Value(long mask)
    {
        long original = ((mask & (1L << proxy)) != 0) ? group : 0L;
        for (int k = 0; k < rest.Length; k++) if ((mask & (1L << k)) != 0) original |= 1L << rest[k];
        return game.Value(original);
    }
    string proxyName = "{" + string.Join(", ", Indices(group, game.N).Select(i => game.Names[i])) + "}";
    return (new CoalitionGameCs(proxy + 1, Value, rest.Select(i => game.Names[i]).Append(proxyName).ToArray()), rest.Append(-1).ToArray());
}
static (double Estimate, double Se) MonteCarloCs(CoalitionGameCs game, long group, int samples, int seed)
{
    Require(group != 0 && samples >= 2, "Groupe non vide et au moins deux échantillons requis.");
    var rng = new Random(seed);
    int m = game.N - BitOperations.PopCount((ulong)group) + 1; double mean = 0, m2 = 0;
    for (int k = 1; k <= samples; k++)
    {
        var rest = Indices(-1L & ~group, game.N);
        int t = rng.Next(m); // .NET PRNG : tirage natif, distinct de NumPy.
        for (int j = rest.Length - 1; j > rest.Length - 1 - t; j--)
        {
            int p = rng.Next(j + 1);
            (rest[j], rest[p]) = (rest[p], rest[j]);
        }
        long predecessor = 0L;
        for (int j = rest.Length - 1; j > rest.Length - 1 - t; j--) predecessor |= 1L << rest[j];
        double x = (game.Value(predecessor | group) - game.Value(predecessor)).ToDouble();
        double delta = x - mean; mean += delta / k; m2 += delta * (x - mean);
    }
    return (mean, Math.Sqrt(m2 / (samples - 1) / samples));
}

class GroupValueTableCs
{
    public CoalitionGameCs Game { get; }
    public int N { get; }
    public Rational[] V { get; }
    public int[] Pop { get; }
    // Valeurs du jeu mises au dénominateur commun : numérateurs entiers exacts, sans réduction par terme.
    private readonly BigInteger[] numerators;
    private readonly BigInteger commonDenominator;
    private readonly BigInteger[] factorials;
    private readonly Dictionary<int, Rational[]> weights = new();
    private readonly Dictionary<int, BigInteger[]> weightNumerators = new();
    public GroupValueTableCs(CoalitionGameCs game)
    {
        Require(game.N <= 22, "Au-delà de 22 joueurs, employer Monte Carlo.");
        Game = game; N = game.N; int size = 1 << N;
        V = new Rational[size]; Pop = new int[size];
        for (int mask = 0; mask < size; mask++) V[mask] = game.Value(mask);
        for (int mask = 0; mask < size; mask++) Pop[mask] = BitOperations.PopCount((uint)mask);
        commonDenominator = V.Aggregate(BigInteger.One, (lcm, v) => lcm / BigInteger.GreatestCommonDivisor(lcm, v.Den) * v.Den);
        numerators = V.Select(v => v.Num * (commonDenominator / v.Den)).ToArray();
        factorials = new BigInteger[N + 1]; factorials[0] = BigInteger.One;
        for (int i = 1; i <= N; i++) factorials[i] = factorials[i - 1] * i;
    }
    private Rational[] W(int n)
    {
        if (!weights.ContainsKey(n)) weights[n] = ShapleyWeights(n);
        return weights[n];
    }
    // Numérateurs des poids de Shapley de m acteurs : s!(m - s - 1)!, le dénominateur commun étant m!.
    private BigInteger[] WeightNumerators(int m)
    {
        if (!weightNumerators.ContainsKey(m))
            weightNumerators[m] = Enumerable.Range(0, m).Select(s => factorials[s] * factorials[m - s - 1]).ToArray();
        return weightNumerators[m];
    }
    public Rational GroupValue(long group)
    {
        int g = checked((int)group);
        Require(g != 0, "Groupe vide.");
        var rest = Indices(V.Length - 1 - g, N); int m = rest.Length + 1;
        var w = WeightNumerators(m);
        // Seuls les sous-ensembles T de N - C ont un poids non nul : 2^(n - |C|) termes au lieu de 2^n.
        // t[sub] est la coalition T du sous-ensemble sub, obtenue en O(1) depuis sub privé de son plus petit bit.
        var bits = rest.Select(i => 1 << i).ToArray();
        var t = new int[1 << rest.Length];
        for (int sub = 1; sub < t.Length; sub++) t[sub] = t[sub ^ (sub & -sub)] | bits[BitOperations.TrailingZeroCount((uint)sub)];
        BigInteger numerator = BigInteger.Zero;
        for (int sub = 0; sub < t.Length; sub++)
        {
            var delta = numerators[t[sub] | g] - numerators[t[sub]];
            if (!delta.IsZero) numerator += w[Pop[sub]] * delta;
        }
        return new Rational(numerator, commonDenominator * factorials[m]);
    }
    public Rational[] Shapley()
    {
        var phi = new Rational[N]; var w = W(N);
        for (int i = 0; i < N; i++)
        {
            int bit = 1 << i; Rational total = R(0);
            for (int s = 0; s < V.Length; s++) if ((s & bit) == 0)
                total += w[Pop[s]] * (V[s | bit] - V[s]);
            phi[i] = total;
        }
        return phi;
    }
    public Rational AdditiveValue(long group, Rational[] phi) => Indices(group, N).Aggregate(R(0), (sum, i) => sum + phi[i]);
    public Rational Profitability(long group, Rational[] phi) => GroupValue(group) - AdditiveValue(group, phi);
    public Rational RestrictedShapley(int player, long removedSet)
    {
        int removed = checked((int)removedSet);
        Require((removed & (1 << player)) == 0, "Le joueur ne peut pas être retiré.");
        int nSub = N - BitOperations.PopCount((uint)removed); var w = W(nSub);
        Rational total = R(0); int bit = 1 << player;
        for (int s = 0; s < V.Length; s++) if ((s & (removed | bit)) == 0)
            total += w[Pop[s]] * (V[s | bit] - V[s]);
        return total;
    }
    public Rational GroupComplementarity(long groupSet, int player)
    {
        int group = checked((int)groupSet);
        Require(group != 0 && (group & (1 << player)) == 0, "Entrant invalide.");
        int m = N - BitOperations.PopCount((uint)group) + 1, bit = 1 << player;
        var w = W(m); Rational total = R(0);
        for (int s = 0; s < V.Length; s++) if ((s & (group | bit)) == 0)
            total += w[Pop[s]] * (V[s | group | bit] - V[s | group] - V[s | bit] + V[s]);
        return total;
    }
    public Rational Complementarity(int i, int j) => GroupComplementarity(1L << i, j);
    public List<(long Group, Rational Value)> BestGroups(int k)
    {
        var result = new List<(long, Rational)>();
        for (int mask = 1; mask < V.Length; mask++) if (Pop[mask] == k) result.Add((mask, GroupValue(mask)));
        return result.OrderByDescending(x => x.Item2).ThenBy(x => x.Item1).ToList();
    }
    public List<int> GreedyGroup(int k)
    {
        var members = new List<int>();
        for (int step = 0; step < k; step++)
        {
            int best = -1; Rational bestValue = R(-1000);
            long group = members.Aggregate(0L, (mask, i) => mask | (1L << i));
            for (int i = 0; i < N; i++) if ((group & (1L << i)) == 0)
            {
                Rational value = GroupValue(group | (1L << i));
                if (best < 0 || value > bestValue) { best = i; bestValue = value; }
            }
            members.Add(best);
        }
        return members;
    }
    public string Names(long group) => string.Join(" + ", Indices(group, N).Select(i => Game.Names[i]));
}

record SvgImage(string Html);
// L'espace Formatting n'est pas importé au niveau du script : sa présence fait passer la recherche
// des méthodes d'extension (Append notamment) par ses classes, et lève un avertissement CS1701
// (identité d'assembly Html.Abstractions 2.3.0.0 contre 10.0.0.0). L'appel est pleinement qualifié.
Microsoft.DotNet.Interactive.Formatting.Formatter.Register<SvgImage>((SvgImage image, TextWriter writer) => writer.Write(image.Html), "text/html");
static string EscapeSvg(string text) => text.Replace("&", "&amp;").Replace("<", "&lt;").Replace(">", "&gt;");
static SvgImage SvgEmbed(string svg)
{
    string encoded = Convert.ToBase64String(Encoding.UTF8.GetBytes(svg));
    return new SvgImage($"<img alt=\"Figure SVG\" src=\"data:image/svg+xml;base64,{encoded}\">");
}
static string SvgOpen(int width, int height, string title) =>
    $"<svg xmlns='http://www.w3.org/2000/svg' viewBox='0 0 {width} {height}' role='img' aria-label='{EscapeSvg(title)}'>" +
    $"<rect width='{width}' height='{height}' fill='white'/><text x='{width / 2}' y='24' text-anchor='middle' font-family='sans-serif' font-size='15' font-weight='bold'>{EscapeSvg(title)}</text>";
static string GraphSvg(int n, (int A, int B)[] edges, double[,] positions, Func<int, string> color, string title)
{
    var sb = new StringBuilder(SvgOpen(700, 360, title));
    foreach (var (a, b) in edges)
        sb.Append(FormattableString.Invariant($"<line x1='{positions[a, 0]:F1}' y1='{positions[a, 1]:F1}' x2='{positions[b, 0]:F1}' y2='{positions[b, 1]:F1}' stroke='#8a949c' stroke-width='1.5'/>"));
    for (int i = 0; i < n; i++)
        sb.Append(FormattableString.Invariant($"<circle cx='{positions[i, 0]:F1}' cy='{positions[i, 1]:F1}' r='16' fill='{color(i)}' stroke='#25313a'/><text x='{positions[i, 0]:F1}' y='{positions[i, 1] + 4:F1}' text-anchor='middle' font-family='sans-serif' font-size='11' fill='#18242c'>{i}</text>"));
    return sb.Append("</svg>").ToString();
}
static int[] Degrees(int n, (int A, int B)[] edges)
{
    var degree = new int[n]; foreach (var (a, b) in edges) { degree[a]++; degree[b]++; } return degree;
}
static double[] Betweenness(int n, (int A, int B)[] edges)
{
    var adj = Enumerable.Range(0, n).Select(_ => new List<int>()).ToArray();
    foreach (var (a, b) in edges) { adj[a].Add(b); adj[b].Add(a); }
    var centrality = new double[n];
    for (int source = 0; source < n; source++)
    {
        var stack = new Stack<int>(); var predecessors = Enumerable.Range(0, n).Select(_ => new List<int>()).ToArray();
        var paths = new double[n]; var distance = Enumerable.Repeat(-1, n).ToArray(); paths[source] = 1; distance[source] = 0;
        var queue = new Queue<int>(); queue.Enqueue(source);
        while (queue.Count > 0)
        {
            int v = queue.Dequeue(); stack.Push(v);
            foreach (int w in adj[v])
            {
                if (distance[w] < 0) { queue.Enqueue(w); distance[w] = distance[v] + 1; }
                if (distance[w] == distance[v] + 1) { paths[w] += paths[v]; predecessors[w].Add(v); }
            }
        }
        var dependency = new double[n];
        while (stack.Count > 0)
        {
            int w = stack.Pop();
            foreach (int v in predecessors[w]) dependency[v] += paths[v] / paths[w] * (1 + dependency[w]);
            if (w != source) centrality[w] += dependency[w];
        }
    }
    double normalization = (n - 1.0) * (n - 2.0);
    return centrality.Select(x => x / normalization).ToArray();
}
static double Spearman(double[] x, double[] y)
{
    Require(x.Length == y.Length, "Séries de tailles différentes.");
    double[] Ranks(double[] values)
    {
        var order = Enumerable.Range(0, values.Length).OrderBy(i => values[i]).ToArray();
        var rank = new double[values.Length];
        for (int start = 0; start < order.Length;)
        {
            int end = start + 1;
            while (end < order.Length && values[order[end]] == values[order[start]]) end++;
            double average = (start + end - 1) / 2.0 + 1;
            for (int k = start; k < end; k++) rank[order[k]] = average;
            start = end;
        }
        return rank;
    }
    var rx = Ranks(x); var ry = Ranks(y); double mx = rx.Average(), my = ry.Average();
    double covariance = 0, vx = 0, vy = 0;
    for (int i = 0; i < x.Length; i++) { covariance += (rx[i] - mx) * (ry[i] - my); vx += (rx[i] - mx) * (rx[i] - mx); vy += (ry[i] - my) * (ry[i] - my); }
    return covariance / Math.Sqrt(vx * vy);
}

Console.WriteLine($"C# / .NET {Environment.Version} | BCL seulement, zéro NuGet");
Console.WriteLine("Rational (BigInteger), jeux de connectivité, Shapley exact et Monte Carlo .NET chargés.");

C# / .NET 10.0.15 | BCL seulement, zéro NuGet


Rational (BigInteger), jeux de connectivité, Shapley exact et Monte Carlo .NET chargés.


## 1. Du joueur au groupe : le jeu fusionné

### 1.1 Définition

Soit un jeu coopératif $(N, v)$ à $n$ joueurs et un groupe $C \subseteq N$ de $c$ joueurs qui s'engagent à agir comme **une seule unité**. Lehrer (1988) remplace le groupe par un **mandataire** unique, noté $c$, et définit le **jeu fusionné** $(N_C, v_C)$ sur $N_C = (N \setminus C) \cup \{c\}$ :

- si le mandataire n'est pas dans $S$, alors $v_C(S) = v(S)$ ;
- si le mandataire est dans $S$, alors $v_C(S) = v\big((S \setminus \{c\}) \cup C\big)$.

La **valeur de Shapley de groupe** est la valeur de Shapley du mandataire dans ce jeu fusionné (définition 2 de l'article) :

$$\varphi^g(C) \;=\; \varphi_c(N_C, v_C) \;=\; \sum_{T \subseteq N \setminus C} \frac{t!\,(m-t-1)!}{m!}\,\big(v(T \cup C) - v(T)\big), \qquad m = n - c + 1.$$

La différence avec la valeur additive $\sum_{i \in C} \varphi_i$ est dans le **nombre d'acteurs** : le groupe arrive en bloc dans un ordre aléatoire de $m$ acteurs, pas de $n$. Il ne compte qu'une fois, et ses membres ne peuvent plus se « voler » mutuellement leurs contributions marginales.

Le code ci-dessous construit le graphe de l'exemple 1 de l'article (neuf sommets, numérotés de 1 à 9 comme dans le texte), le jeu de connectivité d'Amer et Gimenez (2004), $v(S) = 1$ si $S$ induit un sous-graphe connexe avec au moins deux sommets et $0$ sinon, puis le jeu fusionné du groupe $\{4, 5\}$.

In [2]:
// Code 1.1 - Jeu de connectivité de l'exemple 1 et jeu fusionné d'un groupe.
var example1Edges = new[] { (1, 4), (2, 4), (3, 4), (4, 5), (5, 6), (6, 7), (6, 8), (6, 9) }
    .Select(e => (A: e.Item1 - 1, B: e.Item2 - 1)).ToArray();
string[] labels = Enumerable.Range(1, 9).Select(k => k.ToString()).ToArray();
var ex1 = ConnectivityGameCs(9, example1Edges, labels);
var (merged, mapping) = MergingGameCs(ex1, Mask(Idx(4, 5)));
Console.WriteLine("Joueurs du jeu fusionné N_C : ['" + string.Join("', '", merged.Names) + "']");
Console.WriteLine($"n = {ex1.N} joueurs  ->  m = {merged.N} acteurs après fusion de {{4, 5}}");
int proxy = merged.N - 1;
var position = merged.Names.Select((name, i) => (name, i)).ToDictionary(x => x.name, x => x.i);
foreach (var names in new[] { new[] { "{4, 5}" }, new[] { "{4, 5}", "6" }, new[] { "{4, 5}", "7" }, new[] { "7", "8" }, new[] { "6", "7" } })
    Console.WriteLine($"  v_C({string.Join(", ", names)}) = {merged.Value(names.Aggregate(0L, (m, name) => m | (1L << position[name])))}");
var mergedPhi = new GroupValueTableCs(merged).Shapley();
var phiG45 = GroupValueCs(ex1, Mask(Idx(4, 5)));
RequireExact(mergedPhi[proxy], R(187, 280), "Shapley du mandataire");
RequireExact(phiG45, R(187, 280), "Valeur directe du groupe {4,5}");
Console.WriteLine($"\nShapley du mandataire dans (N_C, v_C) : {mergedPhi[proxy]}");
Console.WriteLine($"phi_g({{4, 5}}) par la formule directe   : {phiG45}");

Joueurs du jeu fusionné N_C : ['1', '2', '3', '6', '7', '8', '9', '{4, 5}']


n = 9 joueurs  ->  m = 8 acteurs après fusion de {4, 5}


  v_C({4, 5}) = 1


  v_C({4, 5}, 6) = 1


  v_C({4, 5}, 7) = 0


  v_C(7, 8) = 0


  v_C(6, 7) = 1



Shapley du mandataire dans (N_C, v_C) : 187/280


phi_g({4, 5}) par la formule directe   : 187/280


### Lecture du résultat : le mandataire porte la valeur du groupe

Le jeu fusionné n'a plus que $m = 8$ acteurs : les sept sommets hors du groupe et le mandataire `{4, 5}`. Les cinq évaluations se lisent directement sur le graphe d'origine. Seul, le mandataire vaut 1, car l'arête 4-5 rend le groupe connexe. Avec le sommet 6, il vaut encore 1 (la chaîne 4-5-6). Avec le sommet 7, il vaut 0 : 7 n'est relié à 5 que par 6, absent de la coalition. Hors du groupe, la paire $\{7, 8\}$ vaut 0 (deux feuilles sans leur centre) et la paire $\{6, 7\}$ vaut 1 (une arête).

Les deux calculs de la dernière ligne, la valeur de Shapley du mandataire obtenue par tabulation exacte et la formule directe en fractions, donnent **la même fraction, 187/280**. C'est exactement la valeur publiée par Flores, Molina et Tejada pour ce groupe. La définition 2 n'est donc pas une nouvelle formule mais une **réduction** : évaluer un groupe revient à évaluer un joueur dans un autre jeu.

### 1.2 Pourquoi pas la somme des valeurs individuelles ? L'axiome G-SPB

Le théorème 1 de l'article caractérise $\varphi^g$ comme l'**unique** valeur de groupe qui vérifie quatre propriétés :

| Axiome | Énoncé |
|---|---|
| G-joueur nul | ajouter un joueur nul à un groupe ne change pas sa valeur |
| G-linéarité | la valeur de groupe est linéaire en $v$ |
| G-contributions équilibrées (G-CBC) | l'effet de la présence de $j$ sur l'apport de $i$ au groupe est égal à l'effet de la présence de $i$ sur l'apport de $j$ |
| G-SPB (symétrie dans le marchandage pur) | dans le jeu d'unanimité de la grande coalition $u_N$, tout groupe vaut $1/(n-c+1)$ |

La remarque 2 de l'article montre que si l'on remplace G-SPB par « tout groupe vaut $c/n$ dans $u_N$ », les trois autres axiomes caractérisent la **valeur additive** $\sum_{i \in C} \varphi_i$. Toute la différence entre les deux notions tient donc dans une seule question : dans un marchandage où chacun est indispensable, un groupe de $c$ personnes pèse-t-il comme **une voix** ou comme **$c$ voix** ?

In [3]:
// Code 1.2 - G-SPB contre valeur additive dans le jeu d'unanimité u_N (n = 6).
int nUnanimity = 6;
var unanimité = new CoalitionGameCs(nUnanimity, mask => R(mask == (1L << nUnanimity) - 1 ? 1 : 0));
Console.WriteLine($"{"c",2}  {"phi_g(C)",9}  {"1/(n-c+1)",9}  {"additive",9}  {"c/n",6}");
for (int c = 1; c <= nUnanimity; c++)
{
    long group = (1L << c) - 1;
    var g = GroupValueCs(unanimité, group);
    var a = Indices(group, nUnanimity).Aggregate(R(0), (sum, i) => sum + GroupValueCs(unanimité, 1L << i));
    RequireExact(g, R(1, nUnanimity - c + 1), $"G-SPB c={c}");
    RequireExact(a, R(c, nUnanimity), $"Additive c={c}");
    Console.WriteLine($"{c,2}  {g.ToString(),9}  {R(1, nUnanimity - c + 1).ToString(),9}  {a.ToString(),9}  {R(c, nUnanimity).ToString(),6}");
}

 c   phi_g(C)  1/(n-c+1)   additive     c/n


 1        1/6        1/6        1/6     1/6


 2        1/5        1/5        1/3     1/3


 3        1/4        1/4        1/2     1/2


 4        1/3        1/3        2/3     2/3


 5        1/2        1/2        5/6     5/6


 6          1          1          1       1


### Lecture du résultat : une voix, pas c voix

Pour $c = 1$, les deux notions coïncident (1/6) : un groupe d'une personne est cette personne. Elles divergent aussitôt ensuite. Un groupe de trois joueurs sur six vaut **1/4** au sens de $\varphi^g$ et **1/2** au sens additif : fusionné, il négocie comme un seul acteur face aux trois autres, soit quatre acteurs indispensables à parts égales. Pour $c = 6$, le groupe est la grande coalition et vaut 1 dans les deux cas.

Ce n'est pas un défaut de la valeur additive : elle répond à une autre question (« combien reçoivent au total les membres, pris un par un ? »). Mais quand la question est « que vaut ce groupe **en tant qu'unité** ? », seul $\varphi^g$ tient compte de ce que la fusion retire aux membres : la possibilité de se disputer la même contribution marginale.

### Exercice 1 - Unanimité d'une coalition partielle

Dans le jeu d'unanimité $u_S$ (valeur 1 si et seulement si la coalition contient $S$), la valeur de Shapley individuelle vaut $1/s$ pour chaque membre de $S$ et 0 ailleurs. On cherche la formule de $\varphi^g(C; u_S)$ pour un groupe $C$ quelconque.

- **Étape 1** : avec $n = 7$ et $S = \{0, 1, 2, 3\}$, calculer $\varphi^g(C)$ en fractions exactes pour tous les groupes $C$ de taille 1 à 3.
- **Étape 2** : regrouper les résultats selon $\lvert C \cap S \rvert$ et $\lvert S \setminus C \rvert$.
- **Étape 3** : conjecturer une formule fermée, puis la tester sur $n = 8$ et $S = \{0, 2, 4\}$.

*Indice* : que vaut $v_C$ quand le groupe ne touche pas $S$ ? Combien d'acteurs indispensables restent-ils quand il le touche ?

In [4]:
// Exercice 1 : valeur de groupe dans u_S et formule fermée.
// TODO étudiant : construire u_S avec CoalitionGameCs, puis calculer
// GroupValueCs pour tous les groupes de taille 1 à 3.
Dictionary<long, Rational> resultats = null;  // TODO étudiant : groupes -> fractions
Func<long, long, Rational> formuleConjecturee = null;  // TODO étudiant : (C, S) -> Rational
Console.WriteLine("Exercice 1 à compléter : valeur de groupe dans le jeu d'unanimité u_S.");

Exercice 1 à compléter : valeur de groupe dans le jeu d'unanimité u_S.


## 2. L'exemple 1 de l'article, en arithmétique exacte

### 2.1 Le meilleur binôme n'est pas celui des deux meilleurs individus

Le graphe de l'exemple 1 est une « double étoile » : les feuilles 1, 2, 3 pendent au centre 4, les feuilles 7, 8, 9 au centre 6, et le sommet 5 relie les deux centres. Le code trace ce graphe, calcule les neuf valeurs individuelles en fractions exactes, puis classe les 36 binômes par valeur de groupe.

In [5]:
// Code 2.1 - Valeurs individuelles et classement exact des 36 binômes de l'exemple 1.
var positions1 = new double[9, 2];
var coords1 = new[] { (180, 85), (160, 180), (180, 275), (300, 180), (350, 180), (400, 180), (520, 85), (540, 180), (520, 275) };
for (int i = 0; i < 9; i++) { positions1[i, 0] = coords1[i].Item1; positions1[i, 1] = coords1[i].Item2; }
display(SvgEmbed(GraphSvg(9, example1Edges, positions1,
    i => i == 3 || i == 5 ? "#f4a261" : i == 4 ? "#2a9d8f" : "#e9ecef",
    "Exemple 1 : centres 4 et 6 (orange), pont 5 (vert)")));
var phiExact = Enumerable.Range(0, 9).Select(i => GroupValueCs(ex1, 1L << i)).ToArray();
var scaledPhi = phiExact.Select(p => (int)(p * 360).Num).ToArray();
Require(scaledPhi.SequenceEqual(new[] { -8, -8, -8, 139, 130, 139, -8, -8, -8 }), "Valeurs individuelles de l'exemple 1 incorrectes.");
Console.WriteLine("360 * phi_i : [" + string.Join(", ", scaledPhi) + "]");
var binômes = new List<(int A, int B, Rational G, Rational Add)>();
for (int a = 0; a < 9; a++) for (int b = a + 1; b < 9; b++)
    binômes.Add((a, b, GroupValueCs(ex1, Mask(a, b)), phiExact[a] + phiExact[b]));
var rankedBinômes = binômes.OrderByDescending(x => x.G).ThenBy(x => x.A).ThenBy(x => x.B).ToList();
Console.WriteLine($"\n{"rang",4}  {"binôme",8}  {"phi_g",10}  {"additive",9}");
for (int rank = 0; rank < 5; rank++)
{
    var x = rankedBinômes[rank];
    Console.WriteLine($"{rank + 1,4}  {$"({x.A + 1}, {x.B + 1})",8}  {x.G.ToString(),10}  {Fmt(x.Add.ToDouble()),9}");
}
var expectedPairs = new[] { (3, 4), (4, 5), (3, 5), (0, 3), (1, 3) };
Require(rankedBinômes.Take(5).Select(x => (x.A, x.B)).SequenceEqual(expectedPairs), "Classement des cinq meilleurs binômes.");
RequireExact(rankedBinômes[0].G, R(187, 280), "Meilleur binôme");
RequireExact(rankedBinômes[2].G, R(1, 2), "Binôme des deux centres");
RequireExact(rankedBinômes[3].G, R(131, 280), "Binôme feuille-centre");
int rank46 = rankedBinômes.FindIndex(x => x.A == 3 && x.B == 5) + 1;
Require(rank46 == 3, "Rang du binôme (4,6).");
Console.WriteLine($"\nBinôme des deux meilleurs individus (4, 6) : phi_g = {rankedBinômes[rank46 - 1].G}, rang {rank46} sur 36");

360 * phi_i : [-8, -8, -8, 139, 130, 139, -8, -8, -8]



rang    binôme       phi_g   additive


   1    (4, 5)     187/280     0.7472


   2    (5, 6)     187/280     0.7472


   3    (4, 6)         1/2     0.7722


   4    (1, 4)     131/280     0.3639


   5    (2, 4)     131/280     0.3639



Binôme des deux meilleurs individus (4, 6) : phi_g = 1/2, rang 3 sur 36


### Lecture du résultat : reproduction exacte, et un classement qui s'inverse

Les valeurs individuelles reproduisent **exactement** celles de l'article : $360\,\varphi_i = -8$ pour les six feuilles, $139$ pour les centres 4 et 6, $130$ pour le pont 5. Les feuilles ont une valeur négative : ajoutée à un centre isolé, une feuille crée une paire connexe (+1), mais ajoutée à une coalition connexe qui ne contient pas son centre, elle la déconnecte.

Au classement individuel, le meilleur binôme serait $\{4, 6\}$ (valeur additive $278/360 \approx 0.772$). Au classement par valeur de groupe, il vaut **1/2** et n'arrive qu'au troisième rang (dernière ligne), derrière les binômes centre + pont $\{4, 5\}$ et $\{5, 6\}$ qui valent **187/280**, là encore la valeur publiée. Les deux centres sont redondants : aucun ne rend l'autre plus utile, puisqu'ils ne sont pas adjacents et qu'il faut le pont 5 pour les relier.

### 2.2 Le théorème 2 : indépendance plus complémentarité

La **complémentarité moyenne** de deux joueurs (définition 3) moyenne leur différence d'ordre 2, qui mesure combien la présence de $i$ augmente l'apport de $j$ :

$$\psi_{ij}(N, v) = \sum_{S \subseteq N \setminus \{i,j\}} \frac{s!\,(n-s-1)!}{n!}\,\Delta^2_{ij}(S), \qquad \Delta^2_{ij}(S) = v(S \cup \{i,j\}) - v(S \cup \{i\}) - v(S \cup \{j\}) + v(S).$$

Le **théorème 2** décompose l'apport d'un nouveau membre $i$ à un groupe $C$ en deux termes :

$$\varphi^g(C \cup \{i\}) - \varphi^g(C) = \underbrace{\varphi_i(N \setminus C,\, v_{\vert N \setminus C})}_{\text{valeur de } i \text{ sans le groupe}} + \underbrace{\psi_{ci}(N_C, v_C)}_{\text{complémentarité avec le groupe}}.$$

Le code vérifie cette décomposition sur les deux binômes de l'exemple, avec le centre 4 comme membre initial.

In [6]:
// Code 2.2 - Décomposition du théorème 2 sur l'exemple 1 (groupe initial C = {4}).
var t1 = new GroupValueTableCs(ex1);
Console.WriteLine($"{"binôme",7}  {"phi_4",7}  {"phi_j(N-4)",10}  {"psi_4j",7}  {"somme",7}  {"phi_g",7}");
foreach (int node in new[] { 6, 5 })
{
    int j = Idx(node)[0]; var phi4 = t1.Shapley()[Idx(4)[0]];
    var independent = t1.RestrictedShapley(j, Mask(Idx(4)));
    var complement = t1.Complementarity(Idx(4)[0], j);
    var total = phi4 + independent + complement;
    var group = t1.GroupValue(Mask(Idx(4, node)));
    RequireExact(total, group, $"Théorème 2 pour (4,{node})");
    if (node == 6)
    {
        RequireExact(phi4, R(139, 360), "phi_4"); RequireExact(independent, R(1, 8), "phi_6(N-4)");
        RequireExact(complement, R(-1, 90), "psi_46"); RequireExact(total, R(1, 2), "somme (4,6)");
    }
    else
    {
        RequireExact(independent, R(1, 56), "phi_5(N-4)"); RequireExact(complement, R(19, 72), "psi_45");
        RequireExact(total, R(187, 280), "somme (4,5)");
    }
    Console.WriteLine($"  (4, {node})  {phi4.ToString(),7}  {independent.ToString(),10}  {complement.ToString(),7}  {total.ToString(),7}  {group.ToString(),7}");
}

 binôme    phi_4  phi_j(N-4)   psi_4j    somme    phi_g


  (4, 6)  139/360         1/8    -1/90      1/2      1/2


  (4, 5)  139/360        1/56    19/72  187/280  187/280


### Lecture du résultat : le pont gagne par la complémentarité

Les deux lignes reproduisent les calculs de l'article au terme près : pour le binôme $\{4, 6\}$, $139/360 + 1/8 - 1/90 = 1/2$ ; pour $\{4, 5\}$, $139/360 + 1/56 + 19/72 = 187/280$.

La lecture est instructive. **Sans le centre 4**, le centre 6 vaut encore $1/8$ (il reste le cœur de sa propre étoile) alors que le pont 5 ne vaut plus que $1/56$ : pris isolément, 6 est un bien meilleur renfort. Mais leur **complémentarité** avec 4 est de signe opposé : $\psi_{46} = -1/90$ (les deux centres se font légèrement concurrence), $\psi_{45} = 19/72$ (le pont rend le centre 4 beaucoup plus utile, puisqu'il lui ouvre l'autre moitié du graphe). La complémentarité l'emporte largement sur l'indépendance. C'est exactement l'information que la valeur additive jette.

## 3. Profitabilité et complémentarité

La section 4 de l'article appelle **profitabilité** d'un groupe l'écart entre sa valeur en tant qu'unité et la somme des valeurs de ses membres :

$$P(C) = \varphi^g(C) - \sum_{i \in C} \varphi_i.$$

Un groupe profitable ($P(C) > 0$) gagne à s'intégrer ; un groupe non profitable y perd. Derks et Tijs (2000) et Segal (2003) donnent des conditions suffisantes (portant sur les dividendes de Harsanyi, ou sur le signe des différences d'ordre 3). Le code calcule la matrice des $\psi_{ij}$ et la profitabilité des 36 binômes de l'exemple 1, en séparant les binômes adjacents dans le graphe des autres.

In [7]:
// Code 3.1 - Matrice de complémentarité et profitabilité des binômes de l'exemple 1.
var psi = new Rational[9, 9];
for (int a = 0; a < 9; a++) for (int b = a + 1; b < 9; b++) psi[a, b] = psi[b, a] = t1.Complementarity(a, b);
var phi1 = t1.Shapley();
var adjacent = new List<Rational>(); var nonAdjacent = new List<Rational>();
bool HasEdge1(int a, int b) => example1Edges.Any(e => (e.A == a && e.B == b) || (e.A == b && e.B == a));
for (int a = 0; a < 9; a++) for (int b = a + 1; b < 9; b++)
    (HasEdge1(a, b) ? adjacent : nonAdjacent).Add(t1.Profitability(Mask(a, b), phi1));

// Heatmap SVG : bleu positif, rouge négatif, valeurs imprimées dans les 36 cases hors diagonale.
double maxPsi = Indices((1 << 9) - 1, 9).SelectMany(a => Indices((1 << 9) - 1, 9).Select(b => Math.Abs(psi[a, b].ToDouble()))).Max();
var heat = new StringBuilder(SvgOpen(620, 510, "Complémentarité moyenne psi_ij (exemple 1)"));
const int heatX = 75, heatY = 50, cellSize = 43;
for (int a = 0; a < 9; a++) for (int b = 0; b < 9; b++)
{
    double z = a == b ? 0 : psi[a, b].ToDouble() / maxPsi;
    int red = z >= 0 ? (int)(255 - 190 * z) : 255, blue = z >= 0 ? 255 : (int)(255 + 190 * z);
    string color = $"#{red:X2}{(z >= 0 ? (int)(255 - 110 * z) : (int)(255 + 110 * z)):X2}{blue:X2}";
    heat.Append(FormattableString.Invariant($"<rect x='{heatX + b * cellSize}' y='{heatY + a * cellSize}' width='{cellSize}' height='{cellSize}' fill='{color}' stroke='white'/><text x='{heatX + b * cellSize + cellSize / 2}' y='{heatY + a * cellSize + 25}' text-anchor='middle' font-size='9' font-family='sans-serif'>{(a == b ? "-" : psi[a, b].ToDouble().ToString("0.000"))}</text>"));
}
for (int i = 0; i < 9; i++)
    heat.Append($"<text x='{heatX + i * cellSize + 21}' y='42' text-anchor='middle' font-size='11'>{i + 1}</text><text x='65' y='{heatY + i * cellSize + 26}' text-anchor='end' font-size='11'>{i + 1}</text>");
heat.Append("<text x='265' y='475' font-family='sans-serif' font-size='12'>joueur j</text><text x='16' y='245' transform='rotate(-90 16 245)' font-family='sans-serif' font-size='12'>joueur i</text></svg>");
display(SvgEmbed(heat.ToString()));

(int A, int B) extreme = (0, 0); double maximum = -1;
for (int a = 0; a < 9; a++) for (int b = a + 1; b < 9; b++) if (psi[a, b].ToDouble() > maximum) { maximum = psi[a, b].ToDouble(); extreme = (a, b); }
Require(extreme == (3, 4) && Math.Abs(maximum - 19.0 / 72) < 0.00005, "Maximum de complémentarité.");
Console.WriteLine($"psi max = {Fmt(maximum)} pour la paire ({extreme.A + 1}, {extreme.B + 1})");
extreme = (0, 0); double minimum = 1;
for (int a = 0; a < 9; a++) for (int b = a + 1; b < 9; b++) if (psi[a, b].ToDouble() < minimum) { minimum = psi[a, b].ToDouble(); extreme = (a, b); }
Require(extreme == (0, 5) && Math.Abs(minimum + 37.0 / 360) < 0.00005, "Minimum de complémentarité.");
Console.WriteLine($"psi min = {Fmt(minimum)} pour la paire ({extreme.A + 1}, {extreme.B + 1})");
double minEdges = example1Edges.Min(e => psi[e.A, e.B].ToDouble());
int negativeNonEdges = 0;
for (int a = 0; a < 9; a++) for (int b = a + 1; b < 9; b++) if (!HasEdge1(a, b) && psi[a, b] < R(0)) negativeNonEdges++;
RequireNear(minEdges, 53.0 / 360, 0.00005, "Minimum psi sur arêtes"); Require(negativeNonEdges == 22, "Complémentarités négatives hors arêtes.");
Console.WriteLine($"psi sur les 8 arêtes : min {Fmt(minEdges)} ; binômes non adjacents à psi < 0 : {negativeNonEdges} sur {nonAdjacent.Count}");
foreach (var (name, values) in new[] { ("adjacents", adjacent), ("non adjacents", nonAdjacent) })
{
    double mean = values.Average(x => x.ToDouble()), low = values.Min(x => x.ToDouble()), high = values.Max(x => x.ToDouble());
    int profitable = values.Count(x => x > R(0));
    Console.WriteLine($"Profitabilité des {values.Count,2} binômes {name,-13} : moyenne {Signed(mean)}, min {Signed(low)}, max {Signed(high)}, profitables {profitable}/{values.Count}");
    if (name == "adjacents") Require(profitable == 6 && Math.Abs(mean - 0.0581) < 0.00005 && Math.Abs(low + 0.0794) < 0.00005 && Math.Abs(high - 0.1040) < 0.00005, "Profitabilité adjacente.");
    else Require(profitable == 0 && Math.Abs(mean + 0.0345) < 0.00005 && Math.Abs(low + 0.2722) < 0.00005 && Math.Abs(high + 0.0032) < 0.00005, "Profitabilité non adjacente.");
}
double profit45 = t1.Profitability(Mask(Idx(4, 5)), phi1).ToDouble(), profit14 = t1.Profitability(Mask(Idx(1, 4)), phi1).ToDouble();
RequireNear(profit45, -0.0794, 0.00005, "Profitabilité (4,5)"); RequireNear(profit14, 0.1040, 0.00005, "Profitabilité (1,4)");
Console.WriteLine($"Profitabilité du meilleur binôme (4, 5) : {Signed(profit45)} ; d'un binôme feuille + centre (1, 4) : {Signed(profit14)}");

psi max = 0.2639 pour la paire (4, 5)


psi min = -0.1028 pour la paire (1, 6)


psi sur les 8 arêtes : min 0.1472 ; binômes non adjacents à psi < 0 : 22 sur 28


Profitabilité des  8 binômes adjacents     : moyenne +0.0581, min -0.0794, max +0.1040, profitables 6/8


Profitabilité des 28 binômes non adjacents : moyenne -0.0345, min -0.2722, max -0.0032, profitables 0/28


Profitabilité du meilleur binôme (4, 5) : -0.0794 ; d'un binôme feuille + centre (1, 4) : +0.1040


### Lecture du résultat : le binôme le plus précieux n'est pas profitable

La matrice $\psi$ suit le graphe. Les huit arêtes ont toutes une complémentarité positive, et les deux plus fortes ($19/72 \approx 0.264$) relient le pont 5 aux deux centres. À l'inverse, 22 des 28 binômes non adjacents ont une complémentarité négative ; la plus négative, $(1, 6)$, associe une feuille d'une étoile au centre de l'autre, qui ne peuvent appartenir à une même composante connexe sans 4 et 5. Les six exceptions positives, faibles ($0.022$), sont des feuilles sœurs : elles profitent toutes deux de la présence de leur centre commun.

La profitabilité raconte une autre histoire. Les 28 binômes non adjacents sont tous non profitables : fusionner deux joueurs éloignés leur retire plus qu'il ne leur apporte. Parmi les huit binômes adjacents, six sont profitables, les couples feuille + centre ($+0.104$). Les deux exceptions sont **précisément les deux meilleurs binômes** de la section 2, centre + pont, à $-0.079$ : le groupe le plus précieux perd à s'intégrer, parce que ses deux membres valaient déjà beaucoup séparément. La valeur d'un groupe, sa complémentarité interne et sa profitabilité sont trois grandeurs distinctes. C'est la leçon de Segal (2003), reprise par l'article, et c'est pourquoi un décideur qui veut **le meilleur groupe** doit classer par $\varphi^g$, pas par profitabilité.

### Exercice 2 - Vérifier le théorème 2 sur un groupe de taille 2

Le code 2.2 vérifie le théorème 2 pour un groupe initial d'un seul joueur. On le vérifie maintenant pour $C = \{4, 5\}$ et chaque entrant $i \notin C$.

- **Étape 1** : pour chaque $i \notin \{4, 5\}$, calculer le membre de gauche $\varphi^g(C \cup \{i\}) - \varphi^g(C)$ avec `t1.GroupValue`.
- **Étape 2** : calculer le membre de droite avec `t1.RestrictedShapley(i, C)` et `t1.GroupComplementarity(C, i)`.
- **Étape 3** : afficher l'écart maximal et le meilleur troisième membre. Est-ce le centre 6 ?

*Indice* : `Mask(Idx(4, 5))` construit le masque 0-based du groupe.

In [8]:
// Exercice 2 : théorème 2 pour C = {4, 5} dans l'exemple 1.
// TODO étudiant : boucler sur les entrants i hors du groupe et comparer
// les deux membres de l'identité du théorème 2.
Rational? ecartMax = null;  // TODO étudiant
int? meilleurTroisième = null;  // TODO étudiant
Console.WriteLine("Exercice 2 à compléter : vérification du théorème 2 pour un groupe de taille 2.");

Exercice 2 à compléter : vérification du théorème 2 pour un groupe de taille 2.


## 4. Transposition : le réseau des mariages florentins

L'article applique la méthode à des réseaux clandestins étudiés par Lindelauf, Hamers et Husslage (2013). Ces données (poids de relations tirés de sources d'enquête) ne sont ni reproduites ni utiles ici : le phénomène qui nous intéresse, la redondance entre agents centraux, se lit aussi bien sur un réseau historique public.

Le **réseau des mariages florentins** de Padgett et Ansell (1993) relie les grandes familles de Florence au début du XV$^e$ siècle par des alliances matrimoniales. Il sert classiquement à expliquer l'ascension des Médicis par leur position centrale. La version publiée du réseau compte 15 familles (la famille Pucci, isolée, est exclue) et 20 alliances. On y applique le jeu de connectivité : une coalition de familles vaut 1 si ses alliances la rendent connexe. Avec 15 joueurs, les $2^{15} = 32\,768$ coalitions se tabulent en une fraction de seconde, et tout est calculé **exactement**.

In [9]:
// Code 4.1 - Réseau florentin public : 15 familles, 20 alliances, calcul exact.
var florentineEdgesNamed = new[] {
    ("Acciaiuoli", "Medici"), ("Albizzi", "Ginori"), ("Albizzi", "Guadagni"),
    ("Bischeri", "Guadagni"), ("Castellani", "Barbadori"), ("Castellani", "Peruzzi"),
    ("Castellani", "Strozzi"), ("Guadagni", "Lamberteschi"), ("Medici", "Albizzi"),
    ("Medici", "Barbadori"), ("Medici", "Ridolfi"), ("Medici", "Salviati"),
    ("Medici", "Tornabuoni"), ("Peruzzi", "Bischeri"), ("Peruzzi", "Strozzi"),
    ("Ridolfi", "Tornabuoni"), ("Salviati", "Pazzi"), ("Strozzi", "Bischeri"),
    ("Strozzi", "Ridolfi"), ("Tornabuoni", "Guadagni")
};
var familles = florentineEdgesNamed.SelectMany(e => new[] { e.Item1, e.Item2 }).Distinct().OrderBy(x => x, StringComparer.Ordinal).ToArray();
var rang = familles.Select((name, i) => (name, i)).ToDictionary(x => x.name, x => x.i);
var florentineEdges = florentineEdgesNamed.Select(e => (A: rang[e.Item1], B: rang[e.Item2])).ToArray();
var jeuFlo = ConnectivityGameCs(familles.Length, florentineEdges, familles);
var chronoFlo = Stopwatch.StartNew(); var tFlo = new GroupValueTableCs(jeuFlo); chronoFlo.Stop();
Require(familles.Length == 15 && florentineEdges.Length == 20 && tFlo.V.Length == 32768, "Dimensions du réseau florentin.");
Console.WriteLine($"{familles.Length} familles, {florentineEdges.Length} alliances, {tFlo.V.Length} coalitions tabulées en {chronoFlo.Elapsed.TotalSeconds:F2} s");
var phiFlo = tFlo.Shapley(); var intermediaryFlo = Betweenness(familles.Length, florentineEdges); var degreeFlo = Degrees(familles.Length, florentineEdges);
var ordre = Enumerable.Range(0, familles.Length).OrderByDescending(i => phiFlo[i]).ToArray();
Console.WriteLine($"\n{"famille",13}  {"phi_i",7}  {"degré",5}  {"intermédiarité",14}");
foreach (int i in ordre) Console.WriteLine($"{familles[i],13}  {Fmt(phiFlo[i].ToDouble()),7}  {degreeFlo[i],5}  {Fmt(intermediaryFlo[i], 3),14}");
var expectedFlo = new[] { 0.3463, 0.2539, 0.2418, 0.2035, 0.0425, 0.0398, 0.0377, 0.0286, 0.0241, 0.0167, -0.0002, -0.0179, -0.0644, -0.0688, -0.0835 };
for (int k = 0; k < expectedFlo.Length; k++) RequireNear(phiFlo[ordre[k]].ToDouble(), expectedFlo[k], 0.00005, $"phi {familles[ordre[k]]}");
var expectedNames = new[] { "Medici", "Guadagni", "Albizzi", "Salviati", "Strozzi", "Ridolfi", "Bischeri", "Barbadori", "Tornabuoni", "Castellani", "Peruzzi", "Acciaiuoli", "Lamberteschi", "Ginori", "Pazzi" };
Require(ordre.Select(i => familles[i]).SequenceEqual(expectedNames), "Classement des familles.");
var sumPhiFlo = phiFlo.Aggregate(R(0), (sum, x) => sum + x);
RequireExact(sumPhiFlo, R(1), "Efficacité florentine");
Console.WriteLine($"\nEfficacité : somme des phi_i = {Fmt(sumPhiFlo.ToDouble())} = v(N) = {tFlo.V[tFlo.V.Length - 1]}");

15 familles, 20 alliances, 32768 coalitions tabulées en 0.02 s



      famille    phi_i  degré  intermédiarité


       Medici   0.3463      6           0.522


     Guadagni   0.2539      4           0.255


      Albizzi   0.2418      3           0.212


     Salviati   0.2035      2           0.143


... (8 outputs left out)

 Lamberteschi  -0.0644      1           0.000


       Ginori  -0.0688      1           0.000


        Pazzi  -0.0835      1           0.000



Efficacité : somme des phi_i = 1.0000 = v(N) = 1


### Lecture du résultat : les Médicis en tête, trois familles derrière

Le classement individuel place les **Médicis** nettement en tête ($\varphi \approx 0.346$), suivis des Guadagni ($0.254$), des Albizzi ($0.242$) et des Salviati ($0.204$). Les quatre premiers rangs sont ceux de la centralité d'intermédiarité (colonne de droite), ce qui est attendu : dans le jeu de connectivité, un joueur vaut d'autant plus qu'il est souvent le maillon qui rend une coalition connexe. Le degré, lui, ne suffit pas : les Strozzi ont quatre alliances et ne valent que $0.043$, les Salviati en ont deux et valent $0.204$, parce qu'ils sont le seul accès des Pazzi au reste du réseau. Les quatre familles qui n'ont qu'une alliance ont une valeur négative, comme les feuilles de l'exemple 1. La dernière ligne vérifie l'efficacité : la somme des valeurs individuelles vaut bien $v(N) = 1$.

La question de la section 2 se repose donc ici : **les deux familles à cibler sont-elles les Médicis et les Guadagni ?**

In [10]:
// Code 4.2 - Meilleurs binômes exacts, divergence additive et corrélation de Spearman.
var paires = tFlo.BestGroups(2); // valeur, puis masque : classement déterministe des ex-aequo.
var valeurG = new Dictionary<long, Rational>();
for (int a = 0; a < familles.Length; a++) for (int b = a + 1; b < familles.Length; b++) valeurG[Mask(a, b)] = tFlo.GroupValue(Mask(a, b));
var allPairs = valeurG.Keys.OrderBy(m => m).ToArray();
var addFlo = allPairs.ToDictionary(m => m, m => phiFlo[Indices(m, familles.Length)[0]] + phiFlo[Indices(m, familles.Length)[1]]);
Console.WriteLine($"{"rang",4}  {"binôme",24}  {"phi_g",7}  {"additive",8}");
for (int k = 0; k < 5; k++) Console.WriteLine($"{k + 1,4}  {tFlo.Names(paires[k].Group),24}  {Fmt(paires[k].Value.ToDouble()),7}  {Fmt(addFlo[paires[k].Group].ToDouble()),8}");
var expectedTopFlo = new[] { "Albizzi + Medici", "Medici + Salviati", "Medici + Ridolfi", "Barbadori + Medici", "Medici + Tornabuoni" };
Require(paires.Take(5).Select(x => tFlo.Names(x.Group)).SequenceEqual(expectedTopFlo), "Cinq premiers binômes florentins.");
RequireNear(paires[0].Value.ToDouble(), 0.5161, 0.00005, "Albizzi + Medici");
long bestAdditiveMask = allPairs.OrderByDescending(m => addFlo[m]).ThenBy(m => m).First();
int rankAdditive = paires.FindIndex(x => x.Group == bestAdditiveMask) + 1;
Require(tFlo.Names(bestAdditiveMask) == "Guadagni + Medici" && rankAdditive == 7, "Meilleur binôme additif.");
RequireNear(addFlo[bestAdditiveMask].ToDouble(), 0.6002, 0.00005, "Meilleure valeur additive");
RequireNear(valeurG[bestAdditiveMask].ToDouble(), 0.4102, 0.00005, "Guadagni + Medici phi_g");
Console.WriteLine($"\nMeilleur binôme au sens additif : {tFlo.Names(bestAdditiveMask)} (additive {Fmt(addFlo[bestAdditiveMask].ToDouble())}), rang {rankAdditive} sur {paires.Count} au sens de phi_g ({Fmt(valeurG[bestAdditiveMask].ToDouble())})");
double rho = Spearman(allPairs.Select(m => valeurG[m].ToDouble()).ToArray(), allPairs.Select(m => addFlo[m].ToDouble()).ToArray());
RequireNear(rho, 0.986, 0.0005, "Spearman");
Console.WriteLine($"Corrélation de Spearman entre les deux classements, sur les {paires.Count} binômes : {Fmt(rho, 3)}");
int medicis = rang["Medici"];
Console.WriteLine($"\nThéorème 2 avec C = {{Medici}} (phi_Medici = {Fmt(phiFlo[medicis].ToDouble())}) :");
foreach (string family in new[] { "Albizzi", "Guadagni" })
{
    int j = rang[family]; var independent = tFlo.RestrictedShapley(j, Mask(medicis)); var complement = tFlo.Complementarity(medicis, j);
    var result = phiFlo[medicis] + independent + complement;
    RequireExact(result, tFlo.GroupValue(Mask(medicis, j)), $"Théorème 2 Medici + {family}");
    Console.WriteLine($"  + {family,-9} : phi_j(N - Medici) = {Fmt(independent.ToDouble())}, psi = {Fmt(complement.ToDouble())}  ->  phi_g = {Fmt(result.ToDouble())}");
    RequireNear(independent.ToDouble(), family == "Albizzi" ? 0.0051 : 0.0367, 0.00005, $"Indépendance {family}");
    RequireNear(complement.ToDouble(), family == "Albizzi" ? 0.1647 : 0.0271, 0.00005, $"Complémentarité {family}");
}

rang                    binôme    phi_g  additive


   1          Albizzi + Medici   0.5161    0.5881


   2         Medici + Salviati   0.4711    0.5498


   3          Medici + Ridolfi   0.4548    0.3861


   4        Barbadori + Medici   0.4481    0.3749


   5       Medici + Tornabuoni   0.4394    0.3704



Meilleur binôme au sens additif : Guadagni + Medici (additive 0.6002), rang 7 sur 105 au sens de phi_g (0.4102)


Corrélation de Spearman entre les deux classements, sur les 105 binômes : 0.986



Théorème 2 avec C = {Medici} (phi_Medici = 0.3463) :


  + Albizzi   : phi_j(N - Medici) = 0.0051, psi = 0.1647  ->  phi_g = 0.5161


  + Guadagni  : phi_j(N - Medici) = 0.0367, psi = 0.0271  ->  phi_g = 0.4102


### Lecture du résultat : les Albizzi, pas les Guadagni

Le meilleur binôme au sens de la valeur de groupe est **{Albizzi, Medici}**, avec $\varphi^g \approx 0.516$. Le binôme des deux meilleurs individus, {Guadagni, Medici}, n'arrive qu'au **rang 7** avec $\varphi^g \approx 0.410$, alors que la valeur additive le mettait en tête ($0.600$ contre $0.588$ pour {Albizzi, Medici}).

Le paradoxe n'est pas un effet de bord : les deux classements restent très corrélés sur l'ensemble des 105 binômes (Spearman $\approx 0.99$). Ils ne divergent **qu'en tête**, là où se prend justement la décision. Les rangs 3 à 5 le confirment : les Médicis associés aux Ridolfi, aux Barbadori ou aux Tornabuoni valent chacun plus que {Guadagni, Medici} en tant que groupe, avec une valeur additive inférieure de plus de $0.2$. Le théorème 2 en donne la raison. **Sans les Médicis**, les Albizzi ne valent presque plus rien ($0.005$) alors que les Guadagni gardent de la valeur ($0.037$). Mais la complémentarité avec les Médicis est six fois plus forte pour les Albizzi ($0.165$ contre $0.027$) : les Albizzi sont le passage des Médicis vers les Ginori et les Guadagni, alors que les Guadagni ouvrent sur une région (Bischeri, Lamberteschi, Tornabuoni) que les Médicis atteignent déjà en partie par les Tornabuoni.

In [11]:
// Code 4.3 - Tailles k = 2, 3, 4 : exhaustif, glouton et k meilleurs individus.
Console.WriteLine($"{"k",2}  {"optimum exhaustif",40}  {"phi_g",6}  {"top-k individus phi_g",21}  {"glouton = optimum",17}  {"durée",6}");
foreach (int k in new[] { 2, 3, 4 })
{
    var chrono = Stopwatch.StartNew(); var optimum = tFlo.BestGroups(k)[0];
    long topK = ordre.Take(k).Aggregate(0L, (mask, i) => mask | (1L << i));
    long greedy = tFlo.GreedyGroup(k).Aggregate(0L, (mask, i) => mask | (1L << i));
    chrono.Stop();
    Console.WriteLine($"{k,2}  {tFlo.Names(optimum.Group),40}  {Fmt(optimum.Value.ToDouble()),6}  {Fmt(tFlo.GroupValue(topK).ToDouble()),21}  {(greedy == optimum.Group).ToString(),17}  {chrono.Elapsed.TotalSeconds,5:F2}s");
    Require(greedy == optimum.Group, $"Glouton k={k}");
    RequireNear(optimum.Value.ToDouble(), new[] { 0.0, 0.0, 0.5161, 0.6688, 0.7943 }[k], 0.00005, $"Optimum k={k}");
}
Require(tFlo.BestGroups(3)[0].Group == Mask(rang["Albizzi"], rang["Guadagni"], rang["Medici"]), "Trio optimal.");
Require(tFlo.BestGroups(4)[0].Group == Mask(rang["Albizzi"], rang["Guadagni"], rang["Medici"], rang["Salviati"]), "Quatuor optimal.");

 k                         optimum exhaustif   phi_g  top-k individus phi_g  glouton = optimum   durée


 2                          Albizzi + Medici  0.5161                 0.4102               True   0.03s


 3               Albizzi + Guadagni + Medici  0.6688                 0.6688               True   0.05s


 4    Albizzi + Guadagni + Medici + Salviati  0.7943                 0.7943               True   0.08s


### Lecture du résultat : l'écart se referme avec k, le glouton tient

Pour $k = 2$, l'optimum dépasse nettement le binôme des deux meilleurs individus. Pour $k = 3$, le meilleur trio **coïncide** avec les trois meilleurs individus {Albizzi, Guadagni, Medici} : une fois les Albizzi dans le groupe, les Guadagni redeviennent le meilleur ajout. Pour $k = 4$, l'optimum ajoute les Salviati. Le choix « top-k » n'est donc pas faux en général ; il est faux **pour certains k**, et rien dans le classement individuel ne permet de savoir lesquels.

La construction gloutonne, qui ajoute à chaque étape le joueur de plus grand apport $\varphi^g(C \cup \{i\}) - \varphi^g(C)$ (le membre de gauche du théorème 2), retrouve l'optimum pour chaque $k$ testé sur ce réseau. C'est une heuristique, pas une garantie. La recherche exhaustive reste ici instantanée (colonne de droite) parce que la table des $2^{15}$ valeurs est calculée une seule fois.

In [12]:
// Code 4.4 - Nuage SVG des 105 binômes : valeur additive contre valeur de groupe.
int profitablePairs = allPairs.Count(m => valeurG[m] > addFlo[m]);
Require(profitablePairs == 29, "Nombre de binômes profitables.");
var scatter = new StringBuilder(SvgOpen(720, 480, "Familles florentines : 105 binômes"));
double xmin = Math.Min(allPairs.Min(m => addFlo[m].ToDouble()), allPairs.Min(m => valeurG[m].ToDouble())) - 0.05;
double xmax = Math.Max(allPairs.Max(m => addFlo[m].ToDouble()), allPairs.Max(m => valeurG[m].ToDouble())) + 0.05;
double SX(double x) => 80 + (x - xmin) / (xmax - xmin) * 570;
double SY(double y) => 400 - (y - xmin) / (xmax - xmin) * 340;
scatter.Append(FormattableString.Invariant($"<line x1='80' y1='400' x2='650' y2='400' stroke='#333'/><line x1='80' y1='60' x2='80' y2='400' stroke='#333'/><line x1='{SX(xmin):F1}' y1='{SY(xmin):F1}' x2='{SX(xmax):F1}' y2='{SY(xmax):F1}' stroke='#444' stroke-dasharray='5 4'/>"));
for (int tick = 0; tick <= 5; tick++)
{
    double val = xmin + (xmax - xmin) * tick / 5;
    scatter.Append(FormattableString.Invariant($"<text x='{SX(val):F1}' y='420' text-anchor='middle' font-size='10'>{val:F2}</text><text x='70' y='{SY(val) + 3:F1}' text-anchor='end' font-size='10'>{val:F2}</text>"));
}
foreach (long mask in allPairs)
{
    string color = mask == bestAdditiveMask ? "#e76f51" : mask == paires[0].Group ? "#2a9d8f" : "#adb5bd";
    double radius = mask == bestAdditiveMask || mask == paires[0].Group ? 7 : 3;
    scatter.Append(FormattableString.Invariant($"<circle cx='{SX(addFlo[mask].ToDouble()):F1}' cy='{SY(valeurG[mask].ToDouble()):F1}' r='{radius}' fill='{color}' stroke='#333' stroke-width='0.5'/>"));
}
scatter.Append("<text x='350' y='450' font-family='sans-serif' font-size='13'>valeur additive phi_i + phi_j</text><text x='20' y='230' transform='rotate(-90 20 230)' font-family='sans-serif' font-size='13'>valeur de groupe phi_g({i,j})</text><text x='93' y='82' fill='#2a9d8f' font-size='12'>vert : meilleur phi_g (Albizzi + Medici)</text><text x='93' y='102' fill='#e76f51' font-size='12'>orange : meilleur additif (Guadagni + Medici)</text></svg>");
display(SvgEmbed(scatter.ToString()));
Console.WriteLine($"Binômes profitables (au-dessus de y = x) : {profitablePairs} sur {allPairs.Length}");

Binômes profitables (au-dessus de y = x) : 29 sur 105


### Lecture du résultat : l'ordre global tient, la tête s'inverse

Le nuage suit une tendance croissante très nette, ce qui explique la corrélation de Spearman élevée du code 4.2. La majorité des binômes est sous la diagonale : 29 seulement sur 105 sont profitables (décompte sous la figure), comme dans l'exemple 1 où la fusion coûtait en général. Mais les deux points colorés, à droite du nuage, sont dans le **mauvais ordre** : le point orange (meilleur au sens additif) est plus à droite, le point vert (meilleur au sens de $\varphi^g$) est plus haut. Choisir sur l'axe horizontal, c'est choisir le mauvais binôme.

### Exercice 3 - Connectivité pondérée

Lindelauf et al. (2013) utilisent une variante pondérée du jeu de connectivité : une coalition connexe d'au moins deux joueurs vaut la **somme des poids** de ses membres, $v(S) = \sum_{i \in S} w_i$, et 0 sinon.

- **Étape 1** : construire ce jeu sur le réseau florentin avec $w_i$ = degré de la famille $i$ (nombre d'alliances). On peut réutiliser `jeuFlo.Value(S)` pour tester la connexité.
- **Étape 2** : tabuler avec `GroupValueTableCs`, puis comparer le meilleur binôme exact au binôme des deux meilleurs individus.
- **Étape 3** : le binôme {Albizzi, Medici} reste-t-il optimal ? Expliquer le résultat par le théorème 2.

*Indice* : `CoalitionGameCs(n, mask => ...)` exige $v(\varnothing) = 0$, ce qui est bien le cas ici.

In [13]:
// Exercice 3 : jeu de connectivité pondéré par le degré sur le réseau florentin.
// TODO étudiant : définir les poids, construire le CoalitionGameCs pondéré,
// tabuler avec GroupValueTableCs et comparer BestGroups(2) au top-2 individuel.
Rational[] poids = null;  // TODO étudiant : degrés dans l'ordre de familles
long? meilleurBinômePondere = null;  // TODO étudiant
Console.WriteLine("Exercice 3 à compléter : connectivité pondérée (Lindelauf et al. 2013).");

Exercice 3 à compléter : connectivité pondérée (Lindelauf et al. 2013).


## 5. Quand $2^n$ devient hors de portée : l'estimation Monte Carlo

### 5.1 L'estimateur et sa calibration

La tabulation exacte coûte $2^n$ évaluations de $v$ : elle est confortable pour $n = 15$, impossible pour $n = 34$ ($2^{34} \approx 1.7 \times 10^{10}$). Castro, Gómez et Tejada (2009) estiment la valeur de Shapley par échantillonnage d'ordres aléatoires, et l'article procède ainsi pour ses réseaux.

Pour un groupe, l'estimateur est direct. Dans un ordre uniforme des $m$ acteurs du jeu fusionné, le nombre $t$ de prédécesseurs du mandataire est uniforme sur $\{0, \dots, m-1\}$, et l'ensemble $T$ de ces prédécesseurs est un sous-ensemble uniforme de taille $t$ de $N \setminus C$. La moyenne de $v(T \cup C) - v(T)$ est donc un estimateur **sans biais** de $\varphi^g(C)$, et son écart-type empirique divisé par $\sqrt{n_{\text{échantillons}}}$ donne l'erreur-type.

Avant de s'en servir là où l'on ne peut plus vérifier, on le **calibre** là où l'on connaît la réponse : sur le réseau florentin, pour quatre groupes, trois tailles d'échantillon et vingt graines.

Dans ce carnet, le générateur est `System.Random`, amorcé par une graine entière fixée. L'estimateur est celui qui vient d'être décrit ; ses derniers chiffres et ses durées diffèrent donc de ceux du carnet Python, tandis que les contrôles portent sur les propriétés robustes : couverture, erreur-type et rangs structurants.


In [14]:
// Code 5.1 - Calibration Monte Carlo : PRNG .NET natif, graines entières fixées.
// Les derniers chiffres diffèrent donc de NumPy ; on vérifie convergence et couverture.
var groupesTest = new[] { new[] { "Medici" }, new[] { "Albizzi", "Medici" }, new[] { "Guadagni", "Medici" }, new[] { "Albizzi", "Guadagni", "Medici" } };
int[] sampleSizes = { 250, 1000, 4000 };
var chronoCalibration = Stopwatch.StartNew(); int totalCovered = 0, totalChecks = 0, maxZCount = 0, minCovered = 20; double maxZ = 0, sumZ = 0;
Console.WriteLine($"{"groupe",26}  {"exact",6}  " + string.Join("  ", sampleSizes.Select(s => $"{"n=" + s,17}")));
foreach (var groupNames in groupesTest)
{
    long members = groupNames.Aggregate(0L, (mask, name) => mask | (1L << rang[name]));
    double exact = tFlo.GroupValue(members).ToDouble(); var columns = new List<string>();
    foreach (int size in sampleSizes)
    {
        var standardErrors = new List<double>(); int covered = 0;
        for (int seed = 0; seed < 20; seed++)
        {
            var (estimate, se) = MonteCarloCs(jeuFlo, members, size, seed);
            Require(se > 0 && se < 0.1 && double.IsFinite(estimate), "Erreur-type Monte Carlo invalide.");
            double z = Math.Abs(estimate - exact) / se; maxZ = Math.Max(maxZ, z); sumZ += z; if (z > 3) maxZCount++;
            // 4 SE : borne dure sur 240 estimations ; 3 SE laisserait ~50 % de chance d'une fausse alarme.
            Require(z <= 4, $"Estimation à plus de 4 SE : {string.Join("+", groupNames)}, n={size}, graine={seed}, z={z:F2}.");
            standardErrors.Add(se); if (z < 2) covered++;
        }
        Require(covered >= 15, $"Couverture insuffisante : {covered}/20.");
        totalCovered += covered; totalChecks += 20; minCovered = Math.Min(minCovered, covered);
        columns.Add($"se {standardErrors.Average():F4} | {covered,2}/20");
    }
    Console.WriteLine($"{string.Join(" + ", groupNames),26}  {Fmt(exact),6}  " + string.Join("  ", columns.Select(c => $"{c,17}")));
}
Console.WriteLine($"\nDurée totale : {chronoCalibration.Elapsed.TotalSeconds:F1} s  (se = erreur-type moyenne ; k/20 = graines où |estimation - exact| < 2 se)");
Console.WriteLine($"Couverture totale : {totalCovered}/{totalChecks}, minimum par case {minCovered}/20 ; |z| maximal {maxZ:F2}, moyenne des |z| {sumZ / totalChecks:F3}");
Require(sumZ / totalChecks < 1.2, "Déviation systématique des estimations.");

                    groupe   exact              n=250             n=1000             n=4000


                    Medici  0.3463  se 0.0311 | 18/20  se 0.0155 | 18/20  se 0.0078 | 19/20


          Albizzi + Medici  0.5161  se 0.0323 | 19/20  se 0.0162 | 16/20  se 0.0081 | 20/20


         Guadagni + Medici  0.4102  se 0.0325 | 19/20  se 0.0162 | 20/20  se 0.0081 | 20/20


Albizzi + Guadagni + Medici  0.6688  se 0.0300 | 19/20  se 0.0151 | 20/20  se 0.0076 | 20/20



Durée totale : 0.7 s  (se = erreur-type moyenne ; k/20 = graines où |estimation - exact| < 2 se)


Couverture totale : 228/240, minimum par case 16/20 ; |z| maximal 3.34, moyenne des |z| 0.819


### Lecture du résultat : l'erreur-type est honnête

Deux propriétés se lisent dans le tableau. D'abord, l'erreur-type **se divise par deux quand l'échantillon est multiplié par quatre** (environ $0.03$, puis $0.016$, puis $0.008$) : c'est la décroissance en $1/\sqrt{n}$ attendue. Ensuite, l'intervalle « estimation $\pm$ 2 erreurs-types » contient la valeur exacte dans 228 des 240 essais (95 %) : la case la moins couverte, Albizzi + Medici à 1 000 échantillons, en compte 16 sur 20, toutes les autres au moins 18 sur 20. Le taux nominal d'environ 95 % est respecté sur l'ensemble : l'erreur-type annoncée n'est ni optimiste ni pessimiste.

On peut donc lire une différence entre deux groupes estimés en la rapportant à l'erreur-type de la différence, $\sqrt{se_1^2 + se_2^2}$ pour deux estimations indépendantes. C'est ce qu'on fait maintenant sur un réseau trop grand pour le calcul exact.


### 5.2 Le club de karaté de Zachary

Le **club de karaté de Zachary** (1977) est le réseau d'amitiés des 34 membres d'un club universitaire, observé juste avant sa scission entre l'instructeur (sommet 0) et le président (sommet 33). La table exacte y compterait $2^{34}$ valeurs : `GroupValueTableCs` refuse de la construire au-delà de 22 joueurs. On estime donc tout par Monte Carlo : les 34 valeurs individuelles (via $\varphi^g(\{i\}) = \varphi_i$), puis les binômes formés parmi les huit meilleurs individus.

In [15]:
// Code 5.2 - Club de karaté : estimation .NET des individus et des binômes.
var karateEdges = new[] {
    (0,1),(0,2),(0,3),(0,4),(0,5),(0,6),(0,7),(0,8),(0,10),(0,11),(0,12),(0,13),(0,17),(0,19),(0,21),(0,31),
    (1,2),(1,3),(1,7),(1,13),(1,17),(1,19),(1,21),(1,30),(2,3),(2,7),(2,8),(2,9),(2,13),(2,27),(2,28),(2,32),
    (3,7),(3,12),(3,13),(4,6),(4,10),(5,6),(5,10),(5,16),(6,16),(8,30),(8,32),(8,33),(9,33),(13,33),
    (14,32),(14,33),(15,32),(15,33),(18,32),(18,33),(19,33),(20,32),(20,33),(22,32),(22,33),
    (23,25),(23,27),(23,29),(23,32),(23,33),(24,25),(24,27),(24,31),(25,31),(26,29),(26,33),(27,33),
    (28,31),(28,33),(29,32),(29,33),(30,32),(30,33),(31,32),(31,33),(32,33)
}.Select(e => (A: e.Item1, B: e.Item2)).ToArray();
int nk = 34;
Require(karateEdges.Length == 78 && karateEdges.SelectMany(e => new[] { e.A, e.B }).Distinct().Count() == nk, "Réseau de karaté incomplet.");
var jeuK = ConnectivityGameCs(nk, karateEdges, Enumerable.Range(0, nk).Select(i => i.ToString()).ToArray());
// Notation scientifique à mantisse et exposant séparés : « 1.72e+10 » comme dans le carnet Python.
double twoToNk = Math.Pow(2, nk); int exponent10 = (int)Math.Floor(Math.Log10(twoToNk));
Console.WriteLine($"34 membres, 78 liens, 2^34 = {twoToNk / Math.Pow(10, exponent10):F2}e+{exponent10} coalitions");
var chronoIndividuals = Stopwatch.StartNew();
// 10000 échantillons par sommet : erreur-type d'environ 0.005, pour un classement individuel stable.
var indiv = Enumerable.Range(0, nk).Select(i => MonteCarloCs(jeuK, 1L << i, 10000, 100 + i)).ToArray(); chronoIndividuals.Stop();
var ordreK = Enumerable.Range(0, nk).OrderByDescending(i => indiv[i].Estimate).ToArray();
Console.WriteLine($"Valeurs individuelles estimées en {chronoIndividuals.Elapsed.TotalSeconds:F1} s ; huit premiers :");
Console.WriteLine("  " + string.Join(", ", ordreK.Take(8).Select(i => $"{i}: {indiv[i].Estimate:F3}±{indiv[i].Se:F3}")));
Require(ordreK[0] == 0 && ordreK[1] == 33, "Les deux chefs doivent dominer les valeurs individuelles.");
Require(indiv.All(x => x.Se > 0 && x.Se < 0.03), "Erreurs-types individuelles trop grandes.");
var top8 = ordreK.Take(8).OrderBy(i => i).ToArray(); var estimPaires = new List<(int A, int B, double Est, double Se)>();
var chronoPairs = Stopwatch.StartNew();
for (int ai = 0; ai < top8.Length; ai++) for (int bi = ai + 1; bi < top8.Length; bi++)
{
    int a = top8[ai], b = top8[bi]; var estimate = MonteCarloCs(jeuK, Mask(a, b), 6000, 1000 + 40 * a + b);
    estimPaires.Add((a, b, estimate.Estimate, estimate.Se));
}
estimPaires = estimPaires.OrderByDescending(x => x.Est).ToList(); chronoPairs.Stop();
Console.WriteLine($"\n{estimPaires.Count} binômes estimés en {chronoPairs.Elapsed.TotalSeconds:F1} s ; cinq premiers :");
foreach (var pair in estimPaires.Take(5)) Console.WriteLine($"  ({pair.A}, {pair.B}): phi_g = {pair.Est:F3} ± {pair.Se:F3}");
Require((estimPaires[0].A, estimPaires[0].B) == (0, 33), "Le binôme {0,33} doit être premier.");
Require(estimPaires.All(x => x.Se > 0 && x.Se < 0.02), "Erreur-type des binômes trop grande.");
int thirdIndividual = ordreK[2]; var pair03 = estimPaires.Single(x => x.A == Math.Min(0, thirdIndividual) && x.B == Math.Max(0, thirdIndividual));
int rank03 = estimPaires.IndexOf(pair03) + 1;
Console.WriteLine($"Binôme instructeur + troisième individu ({pair03.A}, {pair03.B}) : phi_g = {pair03.Est:F3} ± {pair03.Se:F3}, rang {rank03} sur {estimPaires.Count}");
int top2Rank = estimPaires.FindIndex(x => (x.A, x.B) == (ordreK.Take(2).Min(), ordreK.Take(2).Max())) + 1;
Console.WriteLine($"Binôme des deux meilleurs individus (0, 33) : rang {top2Rank} sur {estimPaires.Count}");

34 membres, 78 liens, 2^34 = 1.72e+10 coalitions


Valeurs individuelles estimées en 0.4 s ; huit premiers :


  0: 0.509±0.005, 33: 0.228±0.004, 32: 0.084±0.003, 31: 0.065±0.003, 2: 0.062±0.002, 5: 0.048±0.003, 6: 0.042±0.003, 29: 0.034±0.002



28 binômes estimés en 0.2 s ; cinq premiers :


  (0, 33): phi_g = 0.679 ± 0.006


  (0, 31): phi_g = 0.633 ± 0.006


  (0, 2): phi_g = 0.587 ± 0.006


  (0, 32): phi_g = 0.572 ± 0.006


  (0, 5): phi_g = 0.570 ± 0.006


Binôme instructeur + troisième individu (0, 32) : phi_g = 0.572 ± 0.006, rang 4 sur 28


Binôme des deux meilleurs individus (0, 33) : rang 1 sur 28


### Lecture du résultat : ici, les deux chefs forment bien le meilleur binôme

Les deux figures du conflit dominent le classement individuel : l'instructeur (0) vaut environ $0.51$ et le président (33) environ $0.23$, loin devant les six suivants (entre $0.03$ et $0.08$). Parmi les binômes estimés, **{0, 33}** arrive en tête : cette fois, le binôme des deux meilleurs individus est aussi le meilleur binôme. C'est un résultat honnête et utile : la valeur de groupe ne contredit pas systématiquement le classement individuel.

Les rangs suivants sont plus instructifs. Ils associent tous l'instructeur à un autre membre, et l'ordre de ces partenaires ne suit pas leur valeur individuelle : le sommet 32, troisième individu, ne forme avec l'instructeur que le quatrième binôme, derrière {0, 31} et {0, 2}, dont les partenaires valent individuellement moins des quatre cinquièmes de 32. Les écarts de quelques millièmes entre binômes voisins, comme {0, 32} et {0, 5}, restent en revanche inférieurs à l'erreur-type et ne se lisent pas comme un classement. Le sommet 32 n'est pas voisin de l'instructeur, et dix de ses onze autres voisins sont aussi voisins du président : il est redondant avec 33. La section suivante mesure cette redondance au troisième membre.


In [16]:
// Code 5.3 - Troisième membre de {0,33} : la propriété robuste est l'adjacence aux deux chefs.
var chronoTrios = Stopwatch.StartNew(); var trios = new List<(int I, double Est, double Se)>();
for (int i = 0; i < nk; i++) if (i != 0 && i != 33)
{
    var estimate = MonteCarloCs(jeuK, Mask(0, 33, i), 6000, 5000 + i);
    trios.Add((i, estimate.Estimate, estimate.Se));
}
trios = trios.OrderByDescending(x => x.Est).ToList(); chronoTrios.Stop();
bool KarateEdge(int a, int b) => karateEdges.Any(e => (e.A == a && e.B == b) || (e.A == b && e.B == a));
Console.WriteLine($"{trios.Count} candidats estimés en {chronoTrios.Elapsed.TotalSeconds:F1} s ; six meilleurs troisièmes membres :");
foreach (var trio in trios.Take(6))
    Console.WriteLine($"  {trio.I,2}: phi_g({{0, 33, {trio.I}}}) = {trio.Est:F3} ± {trio.Se:F3}  {(KarateEdge(trio.I, 0) && KarateEdge(trio.I, 33) ? "adjacent à 0 et à 33" : "")}");
Require(trios.All(x => x.Se > 0 && x.Se < 0.02), "Erreurs-types des trios trop grandes.");
Require(KarateEdge(trios[0].I, 0) && KarateEdge(trios[0].I, 33), "Le meilleur ajout doit relier les deux chefs.");
var bridges = Enumerable.Range(0, nk).Where(i => i != 0 && i != 33 && KarateEdge(i, 0) && KarateEdge(i, 33)).ToArray();
Require(bridges.SequenceEqual(new[] { 8, 13, 19, 31 }), "Les quatre ponts du réseau public.");
Require(trios.Take(4).Select(x => x.I).OrderBy(i => i).SequenceEqual(bridges), "Les quatre meilleurs ajouts doivent être les ponts.");
int thirdBestIndividual = ordreK.First(i => i != 0 && i != 33);
int thirdRank = trios.FindIndex(x => x.I == thirdBestIndividual) + 1;
var bestTrio = trios[0]; var individualTrio = trios.Single(x => x.I == thirdBestIndividual);
double separation = (bestTrio.Est - individualTrio.Est) / Math.Sqrt(bestTrio.Se * bestTrio.Se + individualTrio.Se * individualTrio.Se);
Require(separation > 5, "La différence entre pont et vedette doit être nettement supérieure au bruit.");
Console.WriteLine($"\nTroisième meilleur individu : {thirdBestIndividual}, rang {thirdRank} sur {trios.Count} comme troisième membre");
Console.WriteLine($"Écart {bestTrio.I} contre {thirdBestIndividual} : {bestTrio.Est - individualTrio.Est:F3}, soit {separation:F1} erreurs-types");

// Layout déterministe en deux camps : positions circulaires, sans simulateur stochastique.
var karatePositions = new double[nk, 2];
for (int i = 0; i < nk; i++)
{
    bool left = i == 0 || karateEdges.Any(e => e.A == 0 && e.B == i);
    int groupIndex = Enumerable.Range(0, nk).Count(j => j < i && (j == 0 || karateEdges.Any(e => e.A == 0 && e.B == j)));
    double angle = 2 * Math.PI * (left ? groupIndex : i - groupIndex) / (left ? 17.0 : 18.0);
    double cx = left ? 245 : 455, cy = 205, rx = left ? 130 : 130, ry = 135;
    if (i == 0) { cx = 250; cy = 205; rx = 0; ry = 0; }
    if (i == 33) { cx = 450; cy = 205; rx = 0; ry = 0; }
    karatePositions[i, 0] = cx + rx * Math.Cos(angle); karatePositions[i, 1] = cy + ry * Math.Sin(angle);
}
display(SvgEmbed(GraphSvg(nk, karateEdges, karatePositions,
    i => i == 0 || i == 33 ? "#e76f51" : i == bestTrio.I ? "#2a9d8f" : i == thirdBestIndividual ? "#f4a261" : "#e9ecef",
    $"Karaté : chefs 0 et 33 (rouge), meilleur ajout {bestTrio.I} (vert), 3e individu {thirdBestIndividual} (orange)")));

32 candidats estimés en 0.2 s ; six meilleurs troisièmes membres :


  31: phi_g({0, 33, 31}) = 0.911 ± 0.004  adjacent à 0 et à 33


  19: phi_g({0, 33, 19}) = 0.813 ± 0.005  adjacent à 0 et à 33


   8: phi_g({0, 33, 8}) = 0.811 ± 0.005  adjacent à 0 et à 33


  13: phi_g({0, 33, 13}) = 0.800 ± 0.005  adjacent à 0 et à 33


   6: phi_g({0, 33, 6}) = 0.735 ± 0.006  


   5: phi_g({0, 33, 5}) = 0.734 ± 0.006  



Troisième meilleur individu : 32, rang 23 sur 32 comme troisième membre


Écart 31 contre 32 : 0.233, soit 32.7 erreurs-types


### Lecture du résultat : le meilleur renfort est un pont, pas une vedette

Le meilleur troisième membre estimé est un membre **adjacent aux deux chefs** : il relie les deux camps, et c'est précisément ce qui manque au binôme {0, 33}, dont les membres ne sont pas voisins. Les quatre premiers candidats sont exactement les quatre membres voisins à la fois de 0 et de 33 (mention en fin de ligne) ; les deux suivants affichés, 6 et 5, voisins du seul instructeur, décrochent de plus de $0.06$. Le troisième meilleur individu, lui, tombe loin dans le classement des troisièmes membres : il apporte au groupe ce que le président lui apporte déjà.

L'écart entre le meilleur ajout et le troisième individu vaut plus de trente erreurs-types (dernière ligne avant la figure) : ce n'est pas du bruit d'échantillonnage. Sur la figure, le nœud vert est relié aux deux nœuds rouges, alors que le nœud orange, relié au seul président, se confond presque avec lui : il partage dix de ses onze autres voisins. C'est le même mécanisme que le pont 5 de l'exemple 1 et que les Albizzi à Florence, cette fois établi sur un réseau où aucun calcul exact n'était possible.

**Portée** : il s'agit d'une étape gloutonne conditionnée à {0, 33}. Le meilleur trio global exigerait d'estimer les $\binom{34}{3} = 5\,984$ trios ; rien ne garantit ici qu'il contienne {0, 33}.


### Exercice 4 (bonus) - Comparer deux groupes avec des nombres aléatoires communs

Pour comparer deux trios $\{0, 33, a\}$ et $\{0, 33, b\}$, les deux estimations du code 5.3 sont indépendantes, et l'erreur-type de leur différence cumule les deux variances. Une technique classique réduit ce bruit : tirer **les mêmes** ordres aléatoires pour les deux groupes et estimer directement la différence.

- **Étape 1** : écrire une fonction qui tire $t$ et le sous-ensemble $T$ **une seule fois** parmi les joueurs hors de $\{0, 33, a, b\}$, puis évalue la contribution des deux groupes sur ce même tirage. Attention : les deux jeux fusionnés n'ont pas le même ensemble de joueurs extérieurs ; une façon correcte de faire consiste à estimer $\varphi^g(\{0,33,a\}) - \varphi^g(\{0,33,b\})$ par le théorème 2 appliqué à $C = \{0, 33\}$.
- **Étape 2** : comparer l'erreur-type obtenue à $\sqrt{se_a^2 + se_b^2}$ pour le même nombre d'échantillons.
- **Étape 3** : combien d'échantillons faut-il pour séparer à 3 erreurs-types les deux meilleurs candidats du code 5.3 ?

In [17]:
// Exercice 4 (bonus) : estimation de la différence entre deux trios par nombres aléatoires communs.
// TODO étudiant : implémenter l'estimateur apparié et comparer les erreurs-types.
double? erreurTypeAppariee = null;  // TODO étudiant
double? erreurTypeIndependante = null;  // TODO étudiant
Console.WriteLine("Exercice 4 (bonus) à compléter : nombres aléatoires communs.");

Exercice 4 (bonus) à compléter : nombres aléatoires communs.


## Résumé

1. **Définition** (section 1) : la valeur de Shapley de groupe $\varphi^g(C)$ est la valeur de Shapley du mandataire du groupe dans le jeu fusionné de Lehrer. Elle diffère de la valeur additive par le seul axiome G-SPB : dans un marchandage pur, un groupe pèse **une voix** ($1/(n-c+1)$), pas **$c$ voix** ($c/n$).
2. **Reproduction exacte** (section 2) : l'exemple 1 de l'article est reproduit en fractions ($360\,\varphi = -8, 139, 130$ ; $\varphi^g(\{4,6\}) = 1/2$ ; $\varphi^g(\{4,5\}) = 187/280$) ainsi que les quatre termes de sa décomposition par le théorème 2. Le meilleur binôme associe un centre et le pont, pas les deux centres.
3. **Complémentarité** (section 3) : la complémentarité $\psi$ est positive sur toutes les arêtes du graphe et négative sur la plupart des paires éloignées ; seuls des binômes adjacents sont profitables, mais les deux meilleurs binômes ne le sont pas. Valeur, complémentarité et profitabilité sont trois grandeurs distinctes.
4. **Florence** (section 4) : sur le réseau de Padgett, calcul exact sur $2^{15}$ coalitions. Le meilleur binôme est {Albizzi, Medici} (0.516) ; le binôme des deux meilleurs individus {Guadagni, Medici} n'est que septième (0.410), alors que la valeur additive les classe dans l'ordre inverse. L'écart disparaît pour $k = 3$.
5. **Monte Carlo** (section 5) : l'estimateur est calibré (erreur-type en $1/\sqrt{n}$, couverture proche du nominal). Sur le club de karaté ($2^{34}$ coalitions), les deux chefs forment bien le meilleur binôme, mais le meilleur troisième membre est un pont entre eux, pas le troisième meilleur individu.

La leçon générale : **classer des individus ne classe pas des groupes**. Quand les agents sont redondants ou complémentaires, il faut évaluer le groupe comme une unité, et le théorème 2 dit exactement quelle information le classement individuel laisse de côté.

## Lectures

- Osborne & Rubinstein, *A Course in Game Theory* (MIT Press, 1994), ch. 14 « Stable Sets, the Bargaining Set, and the Shapley Value », § 14.4 « The Shapley Value » (p. 289) : la valeur de Shapley comme solution axiomatique du jeu coalitionnel — efficacité, symétrie, additivité, joueur nul — dont la valeur de groupe de ce carnet étend l'axiomatique à une coalition traitée comme un seul joueur.

## Références

- **Flores R., Molina E., Tejada J. (2019)**. Evaluating groups with the generalized Shapley value. *4OR* 17(2):141-172. doi:10.1007/s10288-018-0380-8. Article distillé : définition 2, théorèmes 1 et 2, proposition 1, exemple 1.
- **Marichal J.-L., Kojadinovic I., Fujimoto K. (2007)**. Axiomatic characterizations of generalized values. *Discrete Applied Mathematics* 155:26-43. Valeur de Shapley généralisée.
- **Lehrer E. (1988)**. An axiomatization of the Banzhaf value. *International Journal of Game Theory* 17:89-99. Jeu fusionné.
- **Amer R., Gimenez J. M. (2004)**. A connectivity game for graphs. *Mathematical Methods of Operations Research* 60:453-470.
- **Castro J., Gómez D., Tejada J. (2009)**. Polynomial calculation of the Shapley value based on sampling. *Computers & Operations Research* 36:1726-1730.
- **Derks J., Tijs S. (2000)**. On merge properties of the Shapley value. *International Game Theory Review* 2:249-257.
- **Segal I. (2003)**. Collusion, exclusion and inclusion in random-order bargaining. *Review of Economic Studies* 70:439-460.
- **Grabisch M., Roubens M. (1999)**. An axiomatic approach to the concept of interaction among players in cooperative games. *International Journal of Game Theory* 28:547-565.
- **Lindelauf R., Hamers H. J. M., Husslage B. G. M. (2013)**. Cooperative game theoretic centrality analysis of terrorist networks. *European Journal of Operational Research* 229:230-238. Application d'origine de l'article, non reproduite ici.
- **Padgett J. F., Ansell C. K. (1993)**. Robust action and the rise of the Medici, 1400-1434. *American Journal of Sociology* 98(6):1259-1319.
- **Zachary W. W. (1977)**. An information flow model for conflict and fission in small groups. *Journal of Anthropological Research* 33(4):452-473.
- **Port C#** : ce carnet contient les implémentations BCL-only et des assertions pour les valeurs publiées et le théorème 2 ; le carnet Python de référence utilise `cooperative_games/group_value.py` et `tests/test_group_value.py` ; `GameTheory-15-CooperativeGames-Python.ipynb` (valeur de Shapley individuelle) ; `SocialChoice/ChoixSocial-06b-Mobius-Coalitions-Lean-Python.ipynb` (dividendes de Harsanyi).
- **Owen G. (1977)** -- *Values of games with a priori unions*. In: Henn R., Moeschlin O. (eds) Mathematical Economics and Game Theory. Lecture Notes in Economics and Mathematical Systems, vol 141. Springer, Berlin. -> §1 (Du joueur au groupe : le jeu fusionné) : Owen 1977 axiomatise la valeur de Shapley étendue aux "a priori unions" (pré-partitions), dont le jeu fusionné de Lehrer 1988 (cité plus haut) est le cas dégénéré à une seule union. Conceptuellement parent de la "valeur de groupe" Flores-Molina-Tejada 2019 : la fusion v_C est un Owen sur C vu comme singleton de la pré-partition {C, N\C}, ce qui éclaire la définition 2 de l'article 2019.

- **Algaba A., Fragnelli V., Sánchez-Soriano J. (Eds.) (2019)** -- *Handbook of the Shapley Value*. Chapman and Hall / CRC, Boca Raton. -> §2 (L'exemple 1 de l'article, en arithmétique exacte) : l'ouvrage de référence fait l'inventaire des extensions de la valeur de Shapley; le chapitre sur les *generalized values* récapitule la lignée Flores-Molina-Tejada 2019 et ses généralisations (groupes, multi-grappes, jeux à paiements non-transférables), ce qui replace l'exemple 1 du carnet (9 joueurs, exact) dans la cartographie moderne des extensions de la Shapley value.

- **Casajus, A. (2014)** -- *The Shapley value without efficiency and additivity*. Mathematical Social Sciences 68:1-4. doi:10.1016/j.mathsocsci.2013.12.001. -> §3 (Profitabilité et complémentarité) : la profitabilité phi_g(C) - sum(phi_i) relâche l'axiome d'efficiency du Shapley classique en autorisant un groupe à capter plus que la somme de ses membres. Casajus 2014 axiomatise précisément ce relâchement (en l'absence d'efficiency et d'additivity) et identifie les conditions sous lesquelles un mandataire "porte" la valeur d'un groupe qui dépasse la somme -- c'est le fondement axiomatique du signe psi_ij que la cellule Section 3 calcule sur les arêtes du graphe de connectivité.

- **Aadithya S. V., Michalak T. P., Jennings N. R. (2010)** -- *Fast computation of the Shapley value for centrality in networks*. In: Proceedings of the 9th International Conference on Autonomous Agents and Multiagent Systems (AAMAS 2010), Toronto, Canada, pp. 1347-1348. IFAAMAS. -> §4 (Transposition : le réseau des mariages florentins) : premier algorithme d'estimation de la valeur de Shapley de groupe pour la centralité dans les graphes d'influence. Le notebook transpose leur méthode de centralité au réseau florentin, mais en mode **exact** sur 2^15 coalitions (alors que Aadithya 2010 ne traite que des graphes approchés par échantillonnage), ce qui permet de reproduire le classement exact de Padgett-Ansell 1993 et de le comparer à la valeur de groupe phi_g({Albizzi, Medici}).

- **Michalak, T. P., Aadithya, K. V., Szczepanski, P. L., Ravindran, B., & Jennings, N. R. (2013)** -- *Efficient Computation of the Shapley Value for Game-Theoretic Network Centrality*. Journal of Artificial Intelligence Research 46:607-650. doi:10.1613/jair.3806. -> §4+§5 (Florentins et Monte Carlo) : article complet étendant Aadithya 2010 avec garanties théoriques et un passage à l'échelle. Le carnet se place dans la lignée de cette application de la Shapley value de groupe aux réseaux d'influence, et montre la complémentarité entre l'arithmétique exacte (cas 2^15 coalitions, Florentins) et l'estimation Monte Carlo (cas 2^34, club de karaté de Zachary 1977 avec 34 nœuds) -- deux régimes que l'algorithme 2013 unifie.

- **Skibski, O., Michalak, T. P., & Wooldridge, M. (2018)** -- *The Stochastic Shapley Value for coalitional games with externalities*. Games and Economic Behavior 108:65-80. doi:10.1016/j.geb.2017.04.008. -> §3+§4 (Profitabilité et Florentins) : variante de la valeur de Shapley qui, comme la "valeur de groupe" de Flores-Molina-Tejada 2019, mesure la contribution d'une coalition pré-existante. Les deux formalismes coïncident dans le cas unicolore (un seul groupe = N) et ne divergent que sur les coalitions internes. Skibski 2018 propose une "valeur de Shapley stochastique" pour les jeux coalitionnels avec externalités, où la valeur d'une coalition dépend d'un état extérieur aléatoire et où la valeur marginale espérée est réécrite comme une espérance sur les profils de coalition. Cette généralisation est conceptuellement distincte de la "valeur de groupe" Flores-Molina-Tejada 2019, qui opère sans externalité : la convergence entre les deux formalismes ne vaut que dans le cas dégénéré sans externalité et sur la coalition totale, ce qui suffit à utiliser la valeur phi_g(C) du carnet comme proxy prudent de la Shapley stochastique sur le réseau florentin en l'absence d'externalités explicites.
